# AutoSignal: An Agentic Framework for Telemetry Signal Localization

Objective

Turn process telemetry into a defensible session-level triage pipeline with simple baselines, a minimal process-file-parent graph, discovery scoring, and one-anchor steering evaluation.


### Agent Prompts

#### System Prompt

You are the AutoSignal Feature Configuration Agent.

Your job is to inspect an unfamiliar telemetry dataset and select useful columns for malicious-behavior analysis.

Your output will configure a deterministic pipeline that:

1. Normalizes selected columns.
2. Groups rows into behavioral sessions.
3. Creates session-level numeric features.
4. Builds a graph when meaningful relationship columns exist.
5. Evaluates Raw kNN, Structural Statistics, Node2Vec, and GraphSAGE.

You only propose a configuration. You do not preprocess data, execute code, evaluate performance, or declare a winning method.

RULES

- Use column names exactly as provided.
- Never invent a source column.
- Output only valid JSON.
- Select exactly three concise feature sets representing different behavioral hypotheses.
- Select columns because of their behavioral meaning, not because they correlate with labels in the sample.
- Never use labels, red-team annotations, alert results, rule hits, signatures, MITRE mappings, or detector outputs as input features.
- Do not use IDs, hashes, PIDs, GUIDs, ports, timestamps, or categorical codes as continuous numeric measurements.
- IDs may be used for graph construction.
- Timestamps may be used for ordering, sessionization, durations, rates, and timing features.
- Categorical columns may be converted by the engine into counts, unique counts, frequencies, or entropy.
- Raw free text should normally be excluded.
- Set uncertain or unavailable mappings to null.

COLUMN TYPES

Classify selected columns using exactly one of:

- numeric
- boolean
- timestamp
- category

Do not include identifier columns in feature sets. Map them separately for graph construction.

SESSIONIZATION

Select columns representing the stable scope and actor of activity, such as:

- tenant
- endpoint or host
- account or user
- source endpoint
- device
- workload or container

Prefer a composite such as host plus user when both are available.

Do not use resources such as filename, executable, destination address, registry key, or process ID as general session grouping columns. Those describe what the actor interacted with, not the behavioral identity being sessionized.

Missing user_name values reduce owned_by relation coverage and may merge unknown-user processes at the host session level.

The engine, not the agent, chooses the inactivity and maximum-duration thresholds.

FEATURE SETS

Produce exactly three feature sets.

Each feature set must:

- Test a distinct behavioral idea.
- Contain between 2 and 10 source columns when possible.
- Reference only columns declared in selected_columns.
- Be usable without the evaluation label.
- Include a short behavioral rationale.

Choose each feature set's columns first. After selecting the columns, generate a precise name describing the signals actually present.

Do not use the generic names "Temporal and Activity", "Entity and Interaction", or "Combined Behavioral". Do not copy names from the output template.

The engine will determine the exact aggregations from each column's declared type.

SEMANTIC COHERENCE

Every column in a feature set must directly measure the behavior named by that feature set.

Do not add a column merely to ensure every selected column is used, to balance feature-set sizes, or because it could be indirectly related through a speculative explanation.

If a column does not directly support the feature set's behavioral question, move it to a more appropriate feature set or exclude it.

Before returning the JSON, verify:
- Every feature-set column directly matches that set's name and rationale.
- Every feature-set column appears in selected_columns.
- Every selected_column appears in at least one feature set.
- No feature set is padded with weakly related columns.

QUALITY

Provide separate, honest quality readings for:

- behavioral_feature_quality: whether the dataset contains useful behavioral measurements.
- topology_quality: whether meaningful graph connections can be built.

Use exactly one of: high, medium, low, none.

Lack of graph topology must not invalidate otherwise useful behavioral features.

GRAPH CONSTRUCTION

Propose every semantically meaningful typed relationship directly supported by the dataset, not only process-parent and process-file relationships.

Examples include, but are not limited to:

- process parent_of process
- process executes executable
- process loads module
- process touches file
- process runs_on host
- process owned_by user
- user authenticates_to host
- user accesses resource
- host communicates_with network_endpoint
- source_ip communicates_with destination_ip
- flow uses_protocol protocol
- account belongs_to domain
- event associated_with alert_type

A graph relation is valid when each dataset row provides the source and target values for an observed relationship.

For every relation specify:

- A concise snake_case relation name.
- Source node type.
- Exact source column.
- Target node type.
- Exact target column.
- Whether the relationship is naturally directed.
- A short explanation of its meaning.

Node-type names are semantic names and may be created by you. Source and target column names must exist exactly as provided.

When an input row itself should be a node but has no identifier column, use the reserved value "$row_id". The engine will generate a stable row identifier.

The same node type may be constructed from multiple columns. For example, both parent_process_id and process_id may contribute values to the "process" node type.

Do not propose a relation merely because two columns coexist in the same row. The dataset must imply a meaningful interaction, ownership, membership, lineage, execution, access, communication, or association.

Do not create relations from:

- Evaluation labels
- Detector outputs
- Free-form command lines
- Near-unique values with no entity meaning
- Arbitrary numeric measurements



NODE-TYPE CONSISTENCY

Node types must describe the entity represented by the endpoint column. Executable paths and executable hashes must use the node type "executable", not the generic node type "file".



OUTPUT-SCHEMA CONSISTENCY

Use the exact JSON keys shown in the required output schema. For graph direction, always use:

"directed": true

Never substitute alternative keys such as "is_directed".


Return all useful relationships supported by the dataset. If none exist, return an empty graph_relations array. This does not invalidate the non-graph feature sets.


### User Prompt

Analyze this telemetry dataset.

Column types:
{dtypes}

Representative rows:
{sample_rows}

Basic column statistics:
{column_statistics}

Return exactly this JSON structure:

{
  "timestamp_col": null,
  "existing_session_id_col": null,
  "session_group_cols": [],
  "graph_relations": [
      {
      "relation_name": "",
      "source_node_type": "",
      "source_column": "",
      "target_node_type": "",
      "target_column": "",
      "directed": ,
      "meaning": ""
      },
  ],
  "label_col": null,
  "malicious_label_values": [],
  "selected_columns": [
    {
      "column": "exact source column",
      "type": "numeric",
      "reason": "Short explanation of its behavioral value."
    }
  ],
  "feature_sets": [
    {
        "name": "",
        "columns": [],
        "rationale": ""
    },
    {
        "name": "",
        "columns": [],
        "rationale": ""
    },
    {
        "name": "",
        "columns": [],
        "rationale": ""
    }
  ],
  "quality": {
    "behavioral_feature_quality": "medium",
    "topology_quality": "medium",
    "summary": "Short analyst-facing assessment.",
    "limitations": []
  }
}

In [30]:
import warnings
import os
import tempfile
from itertools import combinations

os.environ.setdefault("NUMBA_CACHE_DIR", os.path.join(tempfile.gettempdir(), "numba_cache"))

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import average_precision_score
from umap import UMAP
import json
import datamapplot
from IPython.display import display

warnings.filterwarnings("ignore", message=".*sysctlbyname.*")


In [31]:
SEED = 42
DATA_PATH = "../Data/train-process_uber_summary.parquet"
ROWS = 100_000

RARE_FILE_MIN_DEGREE = 2
RARE_FILE_MAX_DEGREE = 15
SESSION_INACTIVITY_THRESHOLD = pd.Timedelta(minutes=5)
MAX_SESSION_DURATION = pd.Timedelta(minutes=30)

EPOCHS = 8
HIDDEN = 48
OUT = 48
MAX_POS_PER_EDGE_TYPE = 5_000
K = 15
TOP_FRACTIONS = (0.01, 0.05, 0.10)
EXPERIMENT_SEEDS = (42, 43, 44)

np.random.seed(SEED)
torch.manual_seed(SEED)

### Development Dataframe Slice

In [32]:
def load_acme_slice(path=DATA_PATH, rows=ROWS):
    df = pd.read_parquet(path)
    df = df.copy()
    df["_time"] = pd.to_datetime(df["process_started"], errors="coerce", utc=True)
    df = (
        df[df["_time"].notna()]
        .sort_values("_time")
        .drop_duplicates("pid_hash", keep="first")
        .tail(rows)
        .reset_index(drop=True)
    )
    return df


def get_slice_manifest(df):
    return pd.DataFrame([{
        "source_file": os.path.basename(DATA_PATH),
        "row_count": len(df),
        "start_timestamps": (df["_time"].min()),
        "end_timestamps": (df["_time"].max()),
        "num_of_processes": df["pid_hash"].nunique(),
        "num_of_malicious_processes": df["red_team"].fillna(0).astype(int).sum(),
        # "num_of_sessions": df["session_id"].nunique(), # Don't know this since the graph creates sessions based on the connected components of the graph
        # "num_of_malicious_sessions": df["red_team"].fillna(0).astype(int).sum(), # Don't know this since the graph creates sessions based on the connected components of the graph
        "malicious_users_hosts": df[df["red_team"] == 1]["user_name"].drop_duplicates().shape[0] + df[df["red_team"] == 1]["hostname"].drop_duplicates().shape[0],
        "benign_users_hosts": df[df["red_team"] == 0]["user_name"].drop_duplicates().shape[0] + df[df["red_team"] == 0]["hostname"].drop_duplicates().shape[0],
        "session_inactivity_minutes": SESSION_INACTIVITY_THRESHOLD.total_seconds() / 60,
        "max_session_duration_minutes": MAX_SESSION_DURATION.total_seconds() / 60,
        "k": 15,
        "scaler": "StandardScaler",
        "feature_builder": "raw_session_stats",
        "git_commit": "N/A",
    }])

dev_df = load_acme_slice()
dev_df[:50000].to_csv("../artifacts/descriptions/development_slice.csv", index=False)
slice_manifest = get_slice_manifest(dev_df)
slice_manifest.to_csv("../artifacts/descriptions/development_slice_manifest.csv", index=False)

display(slice_manifest)


,source_file,row_count,start_timestamps,end_timestamps,num_of_processes,num_of_malicious_processes,malicious_users_hosts,benign_users_hosts,session_inactivity_minutes,max_session_duration_minutes,k,scaler,feature_builder,git_commit
0,train-process_uber_summary.parquet,100000,2024-09-18 20:45:15.961034+00:00,2024-09-22 23:57:33.487951+00:00,100000,31,2,13,5.0,30.0,15,StandardScaler,raw_session_stats,N/A


In [33]:
dev_df[:20000].to_csv("../artifacts/descriptions/development_slice.csv", index=False)

In [34]:
dev_df

,pid_hash,os_family,agent_id,num_agent_id,hostname,os_pid,process_name,num_process_name,args,num_args,...,lolc_class,lolbas_num_rows,mitre_analytic_ids,mitre_information_domains,mitre_subtypes,mitre_analytic_types,mitre_num_rows,bad_user,red_team,_time
0,0F7F5AC1B974A719B677ED5969708AEA,windows,fe437745-98c3-4d17-8e19-b8ba145caced,1,ACME-DC1,8932,conhost.exe,1,0xffffffff -forcev1,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-18 20:45:15.961034+00:00
1,BF2413CE027AEB9022AA8E232DAD9D8A,windows,fe437745-98c3-4d17-8e19-b8ba145caced,1,ACME-DC1,7380,conhost.exe,1,0xffffffff -forcev1,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-18 20:45:17.073633+00:00
2,3F4619640DE34982708D8A2AC6A81735,windows,fe437745-98c3-4d17-8e19-b8ba145caced,1,ACME-DC1,4172,conhost.exe,1,0xffffffff -forcev1,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-18 20:45:18.185783+00:00
3,5E2F1973949C9472D8FB3A21DB59DA96,windows,fe437745-98c3-4d17-8e19-b8ba145caced,1,ACME-DC1,9704,conhost.exe,1,0xffffffff -forcev1,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-18 20:45:19.295963+00:00
4,9269158EA34ECB01BF5D06E228C3DA2C,windows,fe437745-98c3-4d17-8e19-b8ba145caced,1,ACME-DC1,7784,conhost.exe,1,0xffffffff -forcev1,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-18 20:45:20.521310+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,519C74F7310D49C295AEB7E0A0A55A3E,windows,baa6d56b-daab-414f-bfd5-4ffafb247689,1,EC2AMAZ-R9HHULK,14332,wmic.exe,1,os get version /format:list,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-22 23:56:28.864923+00:00
99996,4576D60FFC275C8ED9FA4E86EF505904,windows,f9ac46c8-0959-4bce-82d9-556a971e7f1a,1,ACME-WS-PLU,3000,wmic.exe,1,computersystem get dnshostname /value,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-22 23:57:33.145198+00:00
99997,1CDE178E919F23289A582CA992E6C02F,windows,f9ac46c8-0959-4bce-82d9-556a971e7f1a,1,ACME-WS-PLU,9116,wmic.exe,1,computersystem get domain /value,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-22 23:57:33.262588+00:00
99998,42474400004FA5634B780D79AAB4BC15,windows,f9ac46c8-0959-4bce-82d9-556a971e7f1a,1,ACME-WS-PLU,10284,wmic.exe,1,os get caption /format:list,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-22 23:57:33.386808+00:00


### Create Holdout Data Slice

In [35]:
holdout_df = load_acme_slice("../Data/test-process_uber_summary.parquet", rows=ROWS)

In [36]:
holdout_df

,pid_hash,os_family,agent_id,num_agent_id,hostname,os_pid,process_name,num_process_name,args,num_args,...,lolc_class,lolbas_num_rows,mitre_analytic_ids,mitre_information_domains,mitre_subtypes,mitre_analytic_types,mitre_num_rows,bad_user,red_team,_time
0,0461D93FFE260EAB781FB3520E7B70B2,windows,baa6d56b-daab-414f-bfd5-4ffafb247689,1,ACME-HH-ZYQ,4412,conhost.exe,1,0xffffffff -forcev1,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-07 17:52:33.408538+00:00
1,89FB41E9F68119288F93842CEE0206D7,windows,baa6d56b-daab-414f-bfd5-4ffafb247689,1,ACME-HH-ZYQ,5532,conhost.exe,1,0xffffffff -forcev1,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-07 17:52:34.766044+00:00
2,A17AD86B3AC98DA0C0021C7C7EF573EB,windows,baa6d56b-daab-414f-bfd5-4ffafb247689,1,ACME-HH-ZYQ,8436,conhost.exe,1,0xffffffff -forcev1,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-07 17:52:36.145088+00:00
3,5AB318329C127176487D2201B4E3746A,windows,baa6d56b-daab-414f-bfd5-4ffafb247689,1,ACME-HH-ZYQ,5456,conhost.exe,1,0xffffffff -forcev1,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-07 17:52:37.469091+00:00
4,D2679C6A0E69AECAFF9E4A6447B2D53B,windows,baa6d56b-daab-414f-bfd5-4ffafb247689,1,ACME-HH-ZYQ,8692,conhost.exe,1,0xffffffff -forcev1,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-07 17:52:38.816364+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,148B2D3EA61BD6CCFE43F5CA32780D40,windows,baa6d56b-daab-414f-bfd5-4ffafb247689,1,ACME-HH-UIR,8276,conhost.exe,1,0xffffffff -forcev1,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-09 06:59:53.690031+00:00
99996,6BA43E6702649C9A6A24870945E2F3DF,windows,baa6d56b-daab-414f-bfd5-4ffafb247689,1,ACME-HH-UIR,5500,conhost.exe,1,0xffffffff -forcev1,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-09 06:59:54.996053+00:00
99997,D6567C5B57EDFFCF69315CB2B4416E19,windows,baa6d56b-daab-414f-bfd5-4ffafb247689,1,ACME-HH-UIR,4660,conhost.exe,1,0xffffffff -forcev1,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-09 06:59:56.361288+00:00
99998,93001F2BE09FA86463BBEC52F0FA6EA2,windows,baa6d56b-daab-414f-bfd5-4ffafb247689,1,ACME-HH-UIR,9508,conhost.exe,1,0xffffffff -forcev1,1,...,NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-09 06:59:57.620498+00:00


### Full Set

In [37]:
# full_df = load_acme_slice(path=DATA_PATH, rows=len(pd.read_parquet(DATA_PATH)))
# full_df

## Parameterized Pipeline Execution

In [38]:
agent_output = """{
  "timestamp_col": "_time",
  "existing_session_id_col": null,
  "session_group_cols": [
    "hostname",
    "user_name"
  ],
  "graph_relations": [
    {
      "relation_name": "parent_of",
      "source_node_type": "process",
      "source_column": "parent_pid_hash",
      "target_node_type": "process",
      "target_column": "pid_hash",
      "directed": true,
      "meaning": "Establishes execution lineage from parent to child process."
    },
    {
      "relation_name": "executes",
      "source_node_type": "process",
      "source_column": "pid_hash",
      "target_node_type": "executable",
      "target_column": "file_sha2",
      "directed": true,
      "meaning": "Links a process to its executable image hash."
    },
    {
      "relation_name": "runs_on",
      "source_node_type": "process",
      "source_column": "pid_hash",
      "target_node_type": "host",
      "target_column": "hostname",
      "directed": true,
      "meaning": "Associates a process with the endpoint where it executed."
    },
    {
      "relation_name": "owned_by",
      "source_node_type": "process",
      "source_column": "pid_hash",
      "target_node_type": "user",
      "target_column": "user_name",
      "directed": true,
      "meaning": "Identifies the user account that owns the executing process."
    }
  ],
  "label_col": "red_team",
  "malicious_label_values": [
    1
  ],
  "selected_columns": [
    {
      "column": "cpu_cycle_count",
      "type": "numeric",
      "reason": "Represents the total CPU cycles consumed, indicating computational intensity."
    },
    {
      "column": "commit_charge",
      "type": "numeric",
      "reason": "Indicates the amount of virtual memory allocated to the process."
    },
    {
      "column": "commit_peak",
      "type": "numeric",
      "reason": "Captures the maximum memory footprint, useful for spotting memory-intensive tasks."
    },
    {
      "column": "duration_seconds",
      "type": "numeric",
      "reason": "Measures the lifespan of the process execution."
    },
    {
      "column": "read_operation_count",
      "type": "numeric",
      "reason": "Counts the number of file system read operations."
    },
    {
      "column": "write_operation_count",
      "type": "numeric",
      "reason": "Counts the number of file system write operations."
    },
    {
      "column": "reg_reads",
      "type": "numeric",
      "reason": "Measures the frequency of registry read operations."
    },
    {
      "column": "reg_writes",
      "type": "numeric",
      "reason": "Measures the frequency of registry write operations."
    },
    {
      "column": "reg_createkeys",
      "type": "numeric",
      "reason": "Indicates structural changes to the registry hierarchy."
    },
    {
      "column": "tcp_connect_count",
      "type": "numeric",
      "reason": "Tracks the number of outbound TCP connections established."
    },
    {
      "column": "tcp_send_size",
      "type": "numeric",
      "reason": "Measures the volume of data transmitted over TCP."
    },
    {
      "column": "tcp_recv_size",
      "type": "numeric",
      "reason": "Measures the volume of data received over TCP."
    },
    {
      "column": "net_total_events",
      "type": "numeric",
      "reason": "Summarizes the overall network activity of the process."
    }
  ],
  "feature_sets": [
    {
      "name": "Process Resource Consumption Profile",
      "columns": [
        "cpu_cycle_count",
        "commit_charge",
        "commit_peak",
        "duration_seconds"
      ],
      "rationale": "Measures computational intensity and memory footprint to identify abnormal resource monopolization, typical in cryptomining or heavy data compression."
    },
    {
      "name": "Local State Modification and Data Access",
      "columns": [
        "read_operation_count",
        "write_operation_count",
        "reg_reads",
        "reg_writes",
        "reg_createkeys"
      ],
      "rationale": "Captures the extent of local state modification and data access, highlighting excessive data staging, persistence mechanisms, or configuration changes."
    },
    {
      "name": "TCP Communication Volume and Directionality",
      "columns": [
        "tcp_connect_count",
        "tcp_send_size",
        "tcp_recv_size",
        "net_total_events"
      ],
      "rationale": "Monitors the volume and directionality of network interactions to identify potential data exfiltration or active command and control channels."
    }
  ],
  "quality": {
    "behavioral_feature_quality": "high",
    "topology_quality": "high",
    "summary": "The dataset provides excellent OS-level behavioral signals including CPU, memory, registry, and network metrics. Graph topology is strong with clear lineage, host, user, and executable image resolution.",
    "limitations": [
      "Network destinations are aggregated and lack specific remote IP/port visibility.",
      "File target paths for I/O operations are missing, preventing file-access graphs."
    ]
  }
}"""

In [39]:
config = json.loads(agent_output)

In [40]:
config.keys()

feature_sets = config["feature_sets"]
graph_relations = config["graph_relations"]

graph_relations

[{'relation_name': 'parent_of',
  'source_node_type': 'process',
  'source_column': 'parent_pid_hash',
  'target_node_type': 'process',
  'target_column': 'pid_hash',
  'directed': True,
  'meaning': 'Establishes execution lineage from parent to child process.'},
 {'relation_name': 'executes',
  'source_node_type': 'process',
  'source_column': 'pid_hash',
  'target_node_type': 'executable',
  'target_column': 'file_sha2',
  'directed': True,
  'meaning': 'Links a process to its executable image hash.'},
 {'relation_name': 'runs_on',
  'source_node_type': 'process',
  'source_column': 'pid_hash',
  'target_node_type': 'host',
  'target_column': 'hostname',
  'directed': True,
  'meaning': 'Associates a process with the endpoint where it executed.'},
 {'relation_name': 'owned_by',
  'source_node_type': 'process',
  'source_column': 'pid_hash',
  'target_node_type': 'user',
  'target_column': 'user_name',
  'directed': True,
  'meaning': 'Identifies the user account that owns the executi

In [41]:
def validate_config(agent_output, df):
    config = json.loads(agent_output)
    # Validate that all selected columns exist in the DataFrame
    selected_columns = [col["column"] for col in config["selected_columns"]]
    missing_columns = [col for col in selected_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing columns in DataFrame: {missing_columns}")
    
    # Validate that the timestamp column exists
    if config["timestamp_col"] not in df.columns:
        raise ValueError(f"Timestamp column '{config['timestamp_col']}' is missing from DataFrame.")
    
    # Validate that the label column exists
    if config["label_col"] not in df.columns:
        raise ValueError(f"Label column '{config['label_col']}' is missing from DataFrame.")
    
    return config

def preprocess_selected_columns(df, config):
    # Select only the columns specified in the config
    selected_columns = [col["column"] for col in config["selected_columns"]]
    selected_columns.append(config["timestamp_col"])  # Ensure timestamp column is included
    selected_columns.append(config["label_col"])  # Ensure label column is included
    return df[selected_columns].copy()

def sessionize(df, config):
    # Placeholder for sessionization logic
    # This function should create sessions based on the timestamp and session_group_cols
    # For now, we will just return the original DataFrame and an empty list of sessions
    return df, []


# Execute
def run_autosignal_pipeline(dev_df):
    config = validate_config(agent_output, dev_df)

    prepared_df = preprocess_selected_columns(dev_df, config)
    sessionized_df, sessions = sessionize(prepared_df, config)

    # TelemetryGraph = 
    # graph = build_graph(sessionized_df, config["graph_relations"])

    # Topology-only methods: run once
    m1_results = run_structural_stats(graph, sessions)
    m2_results = run_node2vec(graph, sessions)

    # Feature-dependent methods: run for each hypothesis
    for hypothesis in config["feature_sets"]:
        columns = hypothesis["columns"]

        session_features = build_session_features(
            sessionized_df,
            sessions,
            columns,
        )
        raw_results = run_raw_knn(session_features)

        process_features = build_process_features(
            sessionized_df,
            columns,
        )
        m3_results = run_graphsage(
            graph,
            process_features,
            sessions,
        )

## Strawman Baseline

In [42]:
# session raw stats -> StandardScaler -> kNN distance anomaly score

In [43]:
def make_strawman_sessions(dev_df):
    df = dev_df.sort_values("process_started").reset_index(drop=True).copy()
    df["process_started"] = pd.to_datetime(df["process_started"], errors="coerce", utc=True)

    user = df["user_name"].astype("string") if "user_name" in df else pd.Series(pd.NA, index=df.index)
    host = df["hostname"].astype("string") if "hostname" in df else pd.Series(pd.NA, index=df.index)
    df["session_identity"] = user.fillna("").where(user.fillna("") != "", "host:" + host.fillna("__missing__"))

    sessions = []
    session_ids = np.empty(len(df), dtype=int)
    next_session_id = 0

    for _, group in df.groupby("session_identity", sort=False):
        current = []
        session_start = None
        prev_time = None

        for idx, row in group.sort_values("process_started").iterrows():
            t = row["process_started"]

            new_session = (
                not current
                or (pd.notna(t) and pd.notna(prev_time) and t - prev_time > SESSION_INACTIVITY_THRESHOLD)
                or (pd.notna(t) and pd.notna(session_start) and t - session_start > MAX_SESSION_DURATION)
            )

            if new_session and current:
                sessions.append(current)
                next_session_id += 1
                current = []
                session_start = t

            if not current:
                session_start = t

            current.append(idx)
            session_ids[idx] = next_session_id
            prev_time = t

        if current:
            sessions.append(current)
            next_session_id += 1

    df["session_id"] = session_ids
    return df, sessions


def assert_canonical_sessions(sessionized_df, sessions, session_table):
    n_processes = len(sessionized_df)
    expected_session_ids = np.arange(len(sessions))

    # Every process has one session ID.
    assert sessionized_df["session_id"].notna().all()
    assert len(sessionized_df) == n_processes

    # Session IDs are unique, contiguous, and consistent.
    observed_ids = np.sort(
        sessionized_df["session_id"].unique()
    )
    assert np.array_equal(observed_ids, expected_session_ids)

    # Every process index appears exactly once in the session lists.
    assigned_indices = np.concatenate([
        np.asarray(indices)
        for indices in sessions
    ])
    assert len(assigned_indices) == n_processes
    assert len(np.unique(assigned_indices)) == n_processes
    assert np.array_equal(
        np.sort(assigned_indices),
        np.arange(n_processes),
    )

    # Each session list agrees with the dataframe mapping.
    for session_id, indices in enumerate(sessions):
        mapped_ids = (
            sessionized_df.loc[indices, "session_id"]
            .to_numpy()
        )
        assert np.all(mapped_ids == session_id)

    # The session table has one ordered row per session.
    assert session_table["session_id"].is_unique
    assert len(session_table) == len(sessions)
    assert np.array_equal(
        session_table["session_id"].to_numpy(),
        expected_session_ids,
    )


def assert_method_output(session_table, scores):
    scores = np.asarray(scores)

    assert scores.ndim == 1
    assert len(scores) == len(session_table)
    assert np.isfinite(scores).all()


def assert_method_alignment(
    session_table,
    method_session_ids,
    scores,
):
    expected_ids = session_table["session_id"].to_numpy()

    assert np.array_equal(
        np.asarray(method_session_ids),
        expected_ids,
    )
    assert_method_output(session_table, scores)


# Numeric columns that are NOT safe to mean/std aggregate as "raw telemetry":
#   - id-like / label-encoded columns: averaging an arbitrary code (e.g. a hash
#     bucket or user-name encoding) has no magnitude meaning, and several of
#     these (num_user_name, num_parent_pid_hash) are identity in disguise,
#     which we will exclude from the baseline.
#   - label_* columns: these appear to be annotation/labeling metadata, not raw
#     telemetry, and risk label-leakage.
#   - sigma-hit and lolbas/mitre columns: these are outputs of *other* detection
#     logic (rule hits, technique matches), not raw process telemetry. Including
#     them would make the "raw stats" baseline informed by another
#     detector's opinion, however, this is worth including later on if we want.
STRAWMAN_EXCLUDE_ID_LIKE = {
    "num_agent_id", "os_pid", "num_process_name", "num_args", "num_user_name",
    "num_parent_pid_hash", "parent_os_pid", "num_parent_os_pid", "num_process_path",
    "num_file_md5", "num_file_sha2", "num_process_start", "num_process_stop",
}
STRAWMAN_EXCLUDE_EXACT = {"lolbas_num_rows", "mitre_num_rows"}


def strawman_numeric_columns(sessionized_df):
    return [
        c for c in sessionized_df.select_dtypes(include=[np.number]).columns
        if c not in {"session_id", "red_team"}
        and not c.startswith("label_")
        and not c.endswith("_sigma_hits")
        and not c.endswith("_sigma_rows")
        and c not in STRAWMAN_EXCLUDE_EXACT
        and c not in STRAWMAN_EXCLUDE_ID_LIKE
    ]


def build_raw_session_stats(sessionized_df):
    rows = []

    numeric_cols = strawman_numeric_columns(sessionized_df)

    for session_id, group in sessionized_df.groupby("session_id", sort=True):
        started = pd.to_datetime(group["process_started"], errors="coerce", utc=True)
        
        row = {
            "session_id": session_id,
            "session_size": len(group),
            "duration": (started.max() - started.min()).total_seconds() if started.notna().any() else 0.0,
            "unique_process_names": group["process_name"].nunique() if "process_name" in group else 0, # capture entropy of process names in a session to encode diversity of activity
            "unique_filenames": group["filename"].nunique() if "filename" in group else 0, # capture entropy of filenames in a session to encode diversity of activity
            "label": "malicious" if "red_team" in group and group["red_team"].fillna(False).astype(bool).any() else "benign",
        }

        for col in numeric_cols:
            row[f"{col}_mean"] = group[col].mean()
            row[f"{col}_std"] = group[col].std()

        rows.append(row)

    return pd.DataFrame(rows).fillna(0)


def knn_distance_anomaly_scores(X, k=15):
    X = np.asarray(X, dtype=np.float32)
    if len(X) <= 1:
        return np.zeros(len(X), dtype=float)

    qk = min(k + 1, len(X))
    dists, _ = cKDTree(X).query(X, k=qk, workers=-1)
    return np.asarray(dists[:, 1:]).mean(axis=1)

def malicious_neighbor_purity_diagnostic(X_scaled, session_table, k=15):
    """Measure whether malicious sessions are locally packed in the fixed representation.

    This is a label-aware representation diagnostic, not a deployable anomaly score.
    Labels are read only after X_scaled has already been built.
    """
    X_scaled = np.asarray(X_scaled, dtype=np.float32)
    is_malicious = (session_table["label"] == "malicious").to_numpy(dtype=bool)
    n_sessions = len(is_malicious)
    n_malicious = int(is_malicious.sum())

    if n_sessions <= 1 or n_malicious == 0:
        return pd.DataFrame([{
            "diagnostic": "malicious_neighbor_purity_at_k",
            "k": 0,
            "n_sessions": n_sessions,
            "n_malicious_sessions": n_malicious,
            "base_malicious_rate_excluding_self": np.nan,
            "malicious_neighbor_purity_at_k": np.nan,
            "median_malicious_neighbor_purity_at_k": np.nan,
            "malicious_sessions_with_any_malicious_neighbor_at_k": np.nan,
            "enrichment_vs_random": np.nan,
        }])

    effective_k = min(k, n_sessions - 1)
    _, indices = cKDTree(X_scaled).query(X_scaled, k=effective_k + 1, workers=-1)
    indices = np.asarray(indices)[:, 1:]
    neighbor_labels = is_malicious[indices]
    malicious_neighbor_purity = neighbor_labels[is_malicious].mean(axis=1)

    base_rate = (n_malicious - 1) / (n_sessions - 1)
    mean_purity = float(malicious_neighbor_purity.mean())

    return pd.DataFrame([{
        "diagnostic": "malicious_neighbor_purity_at_k",
        "k": effective_k,
        "n_sessions": n_sessions,
        "n_malicious_sessions": n_malicious,
        "base_malicious_rate_excluding_self": base_rate,
        "malicious_neighbor_purity_at_k": mean_purity,
        "median_malicious_neighbor_purity_at_k": float(np.median(malicious_neighbor_purity)),
        "malicious_sessions_with_any_malicious_neighbor_at_k": float((neighbor_labels[is_malicious].sum(axis=1) > 0).mean()),
        "enrichment_vs_random": mean_purity / base_rate if base_rate > 0 else np.nan,
    }])

def average_precision(y_true, y_scores):
    """Compute the average precision score for binary classification.
    
    Args: y_true (array-like): True binary labels (0 or 1).
          y_scores (array-like): Target scores. Larger continous kNN distance scores indicate more anomalous sessions. 
    """
    return average_precision_score(y_true, y_scores)

def reviews_to_first_malicious(y_true, y_scores):
    """Compute the number of reviews needed to find the first malicious session.
    
    Args:
        y_true (array-like): True binary labels (0 or 1).
        y_scores (array-like): Target scores. Larger continous kNN distance scores indicate more anomalous sessions.
    """
    y_true = np.asarray(y_true).astype(int)
    y_scores = np.asarray(y_scores).astype(float)

    if y_true.sum() == 0:
        return np.nan
    
    order = np.argsort(-y_scores) # Sort in descending order of scores (more anomalous first)
    ranked_labels = y_true[order]

    first_hit = np.flatnonzero(ranked_labels == 1)  # Index of the first malicious session
    if len(first_hit) == 0:
        return np.nan  # No malicious sessions found
    
    return int(first_hit[0] + 1)  # +1 because we want the count of reviews, not the index

def random_expected_reviews_to_first(y_true):
    """Expected reviews to first malicious session under a random review order.

    For n sessions with m malicious sessions, the expected minimum rank of any
    malicious session under random ordering is (n + 1) / (m + 1).
    """
    y_true = np.asarray(y_true).astype(int)
    n_sessions = len(y_true)
    n_malicious = int(y_true.sum())

    if n_malicious == 0:
        return np.nan

    return (n_sessions + 1) / (n_malicious + 1)

def random_anomaly_scores(n_sessions, seed=SEED):
    """Deterministic random triage scores for a sanity-control baseline."""
    rng = np.random.default_rng(seed)
    return rng.random(n_sessions)

def ranking_at_k_metrics(y_true, y_scores, ks=(25, 50, 100, 250)):
    y_true = np.asarray(y_true).astype(int)
    y_scores = np.asarray(y_scores).astype(float)

    order = np.argsort(-y_scores)
    ranked_labels = y_true[order]

    n_malicious = int(y_true.sum())
    metrics = {}

    for k in ks:
        top_k = min(k, len(ranked_labels))
        found = int(ranked_labels[:top_k].sum())

        metrics[f"found_at_{k}"] = found
        metrics[f"recall_at_{k}"] = found / n_malicious if n_malicious > 0 else np.nan
        metrics[f"precision_at_{k}"] = found / top_k if top_k > 0 else np.nan

    return metrics

session_df, sessions = make_strawman_sessions(dev_df)
session_df

,pid_hash,os_family,agent_id,num_agent_id,hostname,os_pid,process_name,num_process_name,args,num_args,...,mitre_analytic_ids,mitre_information_domains,mitre_subtypes,mitre_analytic_types,mitre_num_rows,bad_user,red_team,_time,session_identity,session_id
0,0F7F5AC1B974A719B677ED5969708AEA,windows,fe437745-98c3-4d17-8e19-b8ba145caced,1,ACME-DC1,8932,conhost.exe,1,0xffffffff -forcev1,1,...,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-18 20:45:15.961034+00:00,host:ACME-DC1,0
1,BF2413CE027AEB9022AA8E232DAD9D8A,windows,fe437745-98c3-4d17-8e19-b8ba145caced,1,ACME-DC1,7380,conhost.exe,1,0xffffffff -forcev1,1,...,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-18 20:45:17.073633+00:00,host:ACME-DC1,0
2,3F4619640DE34982708D8A2AC6A81735,windows,fe437745-98c3-4d17-8e19-b8ba145caced,1,ACME-DC1,4172,conhost.exe,1,0xffffffff -forcev1,1,...,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-18 20:45:18.185783+00:00,host:ACME-DC1,0
3,5E2F1973949C9472D8FB3A21DB59DA96,windows,fe437745-98c3-4d17-8e19-b8ba145caced,1,ACME-DC1,9704,conhost.exe,1,0xffffffff -forcev1,1,...,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-18 20:45:19.295963+00:00,host:ACME-DC1,0
4,9269158EA34ECB01BF5D06E228C3DA2C,windows,fe437745-98c3-4d17-8e19-b8ba145caced,1,ACME-DC1,7784,conhost.exe,1,0xffffffff -forcev1,1,...,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-18 20:45:20.521310+00:00,host:ACME-DC1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,519C74F7310D49C295AEB7E0A0A55A3E,windows,baa6d56b-daab-414f-bfd5-4ffafb247689,1,EC2AMAZ-R9HHULK,14332,wmic.exe,1,os get version /format:list,1,...,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-22 23:56:28.864923+00:00,host:EC2AMAZ-R9HHULK,1084
99996,4576D60FFC275C8ED9FA4E86EF505904,windows,f9ac46c8-0959-4bce-82d9-556a971e7f1a,1,ACME-WS-PLU,3000,wmic.exe,1,computersystem get dnshostname /value,1,...,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-22 23:57:33.145198+00:00,host:ACME-WS-PLU,1243
99997,1CDE178E919F23289A582CA992E6C02F,windows,f9ac46c8-0959-4bce-82d9-556a971e7f1a,1,ACME-WS-PLU,9116,wmic.exe,1,computersystem get domain /value,1,...,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-22 23:57:33.262588+00:00,host:ACME-WS-PLU,1243
99998,42474400004FA5634B780D79AAB4BC15,windows,f9ac46c8-0959-4bce-82d9-556a971e7f1a,1,ACME-WS-PLU,10284,wmic.exe,1,os get caption /format:list,1,...,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0,2024-09-22 23:57:33.386808+00:00,host:ACME-WS-PLU,1243


### Scoring

- KNN-based raw distance to identify and score anomalies, defined in ```knn_distance_anomaly_scores``` - to identify pure anomalies based on raw distance 
- Use ```malicious_neighbor_purity_diagnostic``` as a label-aware representation diagnostic after the raw feature space is fixed.
    - This checks whether malicious sessions are locally packed together in kNN space.
    - It is not a deployable anomaly score because it intentionally reads evaluation labels.

In [44]:
session_table = build_raw_session_stats(session_df)
session_table.loc[session_table["label"] == "malicious"]
feature_columns = [c for c in session_table.columns if c not in {"session_id", "label"}]

In [45]:
umapper = UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    metric='euclidean',
    random_state=SEED
)

embeddings = umapper.fit_transform(session_table[feature_columns].values)

/work/home/mwasti/tuttInstitute/Steering-Telemetry-Triage-with-Self-Supervised-Graph-Geometry/GraphTriage/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [46]:
plot = datamapplot.create_interactive_plot(embeddings, session_table['label'].values, hover_text = session_table['label'].values, title="UMAP Projection of Sessions")
# plot

## Metric Evaluator

In [47]:
def evaluate_precomputed_scores(y_true, y_scores):
    metrics = {
        "average_precision": average_precision(y_true, y_scores),
        "reviews_to_first_malicious": reviews_to_first_malicious(y_true, y_scores),
        "random_expected_reviews_to_first": random_expected_reviews_to_first(y_true),
    }
    metrics["ranking_at_k_metrics"] = ranking_at_k_metrics(y_true, y_scores, ks=(25, 50, 100, 250))
    return metrics

def metric_summary_from_method_results(method_results):
    rows = []

    for method_name, metrics in method_results.items():
        row = {
            "method": method_name,
            "average_precision": metrics.get("average_precision"),
            "reviews_to_first_malicious": metrics.get("reviews_to_first_malicious"),
            "random_expected_reviews_to_first": metrics.get("random_expected_reviews_to_first"),
        }
        row.update(metrics.get("ranking_at_k_metrics", {}))
        rows.append(row)

    return pd.DataFrame(rows)

def triage_metric_eval(features, session_table, k=15):
    """Runs the baseline metric evaluation on the given features and returns the results.
    
    Args:
        features (pd.DataFrame): The features to evaluate - should be directly passable to the scaler and anomaly scoring functions.
        session_table (pd.DataFrame): The session table to update with anomaly scores.
    """

    triage_metrics = {
        "average_precision": None,
        "reviews_to_first_malicious": None,
        "random_expected_reviews_to_first": None,
        "ranking_at_k_metrics": None,
        "raw_distance_scores": None,
        "anomaly_distance_z_scores": None,
        "representation_diagnostic": None,
    }


    y_true = session_table["label"].map({"malicious": 1, "benign": 0}).to_numpy(dtype=int)

    triage_metrics["raw_distance_scores"] = knn_distance_anomaly_scores(features, k)
    session_table["raw_session_stats_knn"] = triage_metrics["raw_distance_scores"]

    triage_metrics["average_precision"] = average_precision(y_true, triage_metrics["raw_distance_scores"])
    triage_metrics["reviews_to_first_malicious"] = reviews_to_first_malicious(y_true, triage_metrics["raw_distance_scores"])
    triage_metrics["random_expected_reviews_to_first"] = random_expected_reviews_to_first(y_true)
    triage_metrics["ranking_at_k_metrics"] = ranking_at_k_metrics(y_true, triage_metrics["raw_distance_scores"], ks=(25, 50, 100, 250))

    # The distance ranking is our anomaly score, so we can sort the sessions by this score to identify the most anomalous sessions
    display(session_table["raw_session_stats_knn"].describe()) # Showing massive outliers, the max is ~100x the 75th percentile
    display(session_table["raw_session_stats_knn"].sort_values(ascending=False).head(10)) # Massive distances between top sessions, meaning there are distinct outliers

    # To make this more human interpretable, we will use Z-scores to identify which features are driving the anomaly score for the top anomalous session.
    triage_metrics["anomaly_distance_z_scores"] = pd.DataFrame((triage_metrics["raw_distance_scores"] - np.mean(triage_metrics["raw_distance_scores"])) / np.std(triage_metrics["raw_distance_scores"]), columns=["anomaly_distance_z_scores"])
    display(triage_metrics["anomaly_distance_z_scores"])

    # Label-aware representation diagnostic: are malicious sessions packed together in this feature space?
    triage_metrics["representation_diagnostic"] = malicious_neighbor_purity_diagnostic(features, session_table, k)
    display(triage_metrics["representation_diagnostic"])

    return triage_metrics

## Telemetry Graph Builder

The graph keeps process, file, parent-child, and temporal relations. Evaluation sessions use the same identity rule and split after five minutes of inactivity or 30 minutes total duration, preventing transitive multi-day components.

In [48]:
class TelemetryGraphBuilder:
    FEATURE_COLS = [
        # "duration_seconds",
        # "num_uniq_file_hash",
        # "net_total_events",
        # "conn_id_count",
        # "reg_totals",
    ]

    FEATURE_NAMES = [
        # "duration",
        # "num_file_hash",
        # "net_events",
        # "conn_id_count",
        # "reg_totals",
    ]

    def __init__(
        self,
        rare_file_min_degree=RARE_FILE_MIN_DEGREE,
        rare_file_max_degree=RARE_FILE_MAX_DEGREE,
    ):
        self.rare_file_min_degree = rare_file_min_degree
        self.rare_file_max_degree = rare_file_max_degree


    def build(self, sessionized_df, feature_cols=None):
        assert "session_id" in sessionized_df.columns
        assert sessionized_df["session_id"].notna().all()
        assert sessionized_df["pid_hash"].is_unique
    
        self.df = sessionized_df.reset_index(drop=True).copy()
        self.feature_cols = feature_cols 
        
        self.process_ids = self.df["pid_hash"].tolist()

        # Create a mapping from process IDs to their corresponding indices in the graph
        self.node_index = {pid: i for i, pid in enumerate(self.process_ids)}
        self.executable_index = {}
        
        # Mapping from executables to the processes that touch them
        self.executable_to_processes = {}

        # Contain Parent to Child Edges and Executable Touches Edges
        self.parent_child_edges = []
        self.executes_edges = []

        # Add edges for parent-child relationships, and executable touches
        self._add_parent_child_edges()
        self._add_executable_edges()

        data = self._to_heterodata()

        assert data["process"].num_nodes == len(self.df)

        assert np.array_equal(
            data["process"].session_id.cpu().numpy(),
            self.df["session_id"].to_numpy(),
        )

        return data


    def _process_features(self):
        if not self.feature_cols:
            raise ValueError("No feature columns specified for process features.")
        X = self.df[self.feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=np.float32)
        return torch.tensor(X, dtype=torch.float)

    # def _process_labels(self):
    #     y = pd.to_numeric(self.df["red_team"], errors="coerce").fillna(0).astype(int).to_numpy()
    #     return torch.tensor(y, dtype=torch.long)

    def _add_parent_child_edges(self):
        for row in self.df.itertuples(index=False):
            parent = getattr(row, "parent_pid_hash")
            child = getattr(row, "pid_hash")
            if pd.notna(parent) and parent in self.node_index and child in self.node_index:
                self.parent_child_edges.append((self.node_index[parent], self.node_index[child]))

    # def _add_file_edges(self):
    #     file_counts = self.df["filename"].dropna().value_counts()
    #     rare_files = set(file_counts[
    #         (file_counts >= self.rare_file_min_degree) &
    #         (file_counts <= self.rare_file_max_degree)
    #     ].index)

    #     for row in self.df.itertuples(index=False):
    #         fname = getattr(row, "filename")
    #         pid = getattr(row, "pid_hash")
    #         if pd.isna(fname) or fname not in rare_files or pid not in self.node_index:
    #             continue
    #         if fname not in self.file_index:
    #             self.file_index[fname] = len(self.file_index)
    #         pidx = self.node_index[pid]
    #         fidx = self.file_index[fname]
    #         self.executes_edges.append((pidx, fidx))
    #         self.file_to_processes.setdefault(fname, []).append(pidx)

    def _add_executable_edges(self):
        executable_counts = self.df["filename"].dropna().value_counts()

        shared_executables = set(
            executable_counts[executable_counts >= 2].index
        )

        for row in self.df.itertuples(index=False):
            fname = getattr(row, "filename")
            pid = getattr(row, "pid_hash")

            if (
                pd.isna(fname)
                or fname not in shared_executables
                or pid not in self.node_index
            ):
                continue

            if fname not in self.executable_index:
                self.executable_index[fname] = len(self.executable_index)

            process_index = self.node_index[pid]
            executable_index = self.executable_index[fname]

            self.executes_edges.append(
                (process_index, executable_index)
            )

    def _edge_tensor(self, edges):
        return torch.tensor(edges, dtype=torch.long).t().contiguous()

    def _to_heterodata(self):
        data = HeteroData()
        data["process"].x = self._process_features()
        data["process"].session_id = torch.tensor(
            self.df["session_id"].to_numpy(),
            dtype=torch.long
        )

        # data["process"].y = self._process_labels()
        data["executable"].x = torch.ones((len(self.executable_index), 1), dtype=torch.float)

        empty_edges = torch.empty((2, 0), dtype=torch.long) # Create an empty edge tensor for cases where there are no edges of a certain type

        parent_ei = self._edge_tensor(self.parent_child_edges) if self.parent_child_edges else empty_edges # Create an edge tensor for parent-child edges if they exist, otherwise use the empty edge tensor

        data["process", "parent_of", "process"].edge_index = parent_ei
        data["process", "child_of", "process"].edge_index = parent_ei.flip(0)
        
        touches_ei = self._edge_tensor(self.executes_edges) if self.executes_edges else empty_edges # Create an edge tensor for touches edges if they exist, otherwise use the empty edge tensor
        # data["process", "touches", "executables"].edge_index = touches_ei
        # data["executables", "touched_by", "process"].edge_index = touches_ei.flip(0)
        
        data["process", "executes", "executable"].edge_index = touches_ei
        data["executable", "executed_by", "process"].edge_index = touches_ei.flip(0)

        return data

#### Test

In [49]:
# graph_data['process']

In [50]:
builder = TelemetryGraphBuilder()

sessionized_df, sessions = make_strawman_sessions(dev_df)
session_table = build_raw_session_stats(sessionized_df)
feature_columns = strawman_numeric_columns(sessionized_df)

graph_data = builder.build(sessionized_df, feature_cols=feature_columns)

assert graph_data["process"].num_nodes == len(sessionized_df)
assert len(graph_data["process"].session_id) == len(sessionized_df)

process_df = dev_df.set_index("pid_hash").reindex(builder.process_ids).reset_index()

session_sizes = np.asarray([len(session) for session in sessions])
session_spans = np.asarray([
    (
        process_df.loc[session, "_time"].max()
        - process_df.loc[session, "_time"].min()
    ).total_seconds() / 60.0
    for session in sessions
])

session_sizes = np.asarray([len(session) for session in sessions])
session_spans = np.asarray([
    (
        process_df.loc[session, "_time"].max()
        - process_df.loc[session, "_time"].min()
    ).total_seconds() / 60.0
    for session in sessions
])

build_summary = pd.DataFrame([{
    "process_nodes": graph_data["process"].num_nodes,
    "executable_nodes": graph_data["executable"].num_nodes,
    "edge_types": len(graph_data.edge_types),
    "sessions": len(sessions),
    # "malicious_sessions": int(sum(any(graph_data["process"].y[n].item() == 1 for n in s) for s in sessions)),
    "median_session_size": float(np.median(session_sizes)),
    "p95_session_size": float(np.quantile(session_sizes, 0.95)),
    "max_session_size": int(session_sizes.max()),
    "p95_session_minutes": float(np.quantile(session_spans, 0.95)),
    "max_session_minutes": float(session_spans.max()),
    "parent_child_edges": len(builder.parent_child_edges),
    "executes_edges": len(builder.executes_edges),
}])

display(build_summary)

,process_nodes,executable_nodes,edge_types,sessions,median_session_size,p95_session_size,max_session_size,p95_session_minutes,max_session_minutes,parent_child_edges,executes_edges
0,100000,53,4,1457,76.0,147.0,213,29.999298,29.999997,2318,99850


In [51]:
parent_ei = graph_data[
    "process", "parent_of", "process"
].edge_index

touches_ei = graph_data[
    "process", "executes", "executable"
].edge_index

incident_processes = torch.unique(torch.cat([
    parent_ei.reshape(-1),
    touches_ei[0],
]))

covered_session_ids = torch.unique(
    graph_data["process"].session_id[incident_processes]
)

session_coverage = pd.DataFrame([{
    "total_sessions": len(session_table),
    "sessions_with_graph_evidence": len(covered_session_ids),
    "session_coverage": (
        len(covered_session_ids) / len(session_table)
    ),
    "processes_with_graph_evidence": len(incident_processes),
    "process_coverage": (
        len(incident_processes)
        / graph_data["process"].num_nodes
    ),
}])

session_coverage

,total_sessions,sessions_with_graph_evidence,session_coverage,processes_with_graph_evidence,process_coverage
0,1457,1457,1.0,100000,1.0


In [52]:
graph_data

HeteroData(
  process={
    x=[100000, 67],
    session_id=[100000],
  },
  executable={ x=[53, 1] },
  (process, parent_of, process)={ edge_index=[2, 2318] },
  (process, child_of, process)={ edge_index=[2, 2318] },
  (process, executes, executable)={ edge_index=[2, 99850] },
  (executable, executed_by, process)={ edge_index=[2, 99850] }
)

In [53]:
executes_ei = graph_data[
    "process", "executes", "executable"
].edge_index

executable_degrees = torch.bincount(
    executes_ei[1],
    minlength=graph_data["executable"].num_nodes,
).cpu().numpy()

process_executable_degrees = torch.bincount(
    executes_ei[0],
    minlength=graph_data["process"].num_nodes,
).cpu().numpy()

telemetry_graph_summary = pd.DataFrame([{
    "process_nodes": graph_data["process"].num_nodes,
    "executable_nodes": graph_data["executable"].num_nodes,
    "parent_edges": graph_data[
        "process", "parent_of", "process"
    ].edge_index.shape[1],
    "executes_edges": executes_ei.shape[1],
    "session_coverage": 1.0,
    "executable_degree_median": np.median(executable_degrees),
    "executable_degree_p90": np.quantile(executable_degrees, 0.90),
    "executable_degree_max": executable_degrees.max(),
    "process_executable_degree_median": np.median(
        process_executable_degrees
    ),
}])

telemetry_graph_summary

,process_nodes,executable_nodes,parent_edges,executes_edges,session_coverage,executable_degree_median,executable_degree_p90,executable_degree_max,process_executable_degree_median
0,100000,53,2318,99850,1.0,32.0,491.2,74565,1.0


### Stage 2

#### M1 - Typed Structural Statistics

In [54]:
def build_process_structural_features(graph_data):
    n_processes = graph_data["process"].num_nodes
    n_executables = graph_data["executable"].num_nodes
    
    parent_ei = graph_data["process", "parent_of", "process"].edge_index.cpu()
    executes_ei = graph_data["process", "executes", "executable"].edge_index.cpu()

    # Typed process-lineage degeres
    parent_out_degree = torch.bincount(parent_ei[0], minlength=n_processes).float()
    parent_in_degree = torch.bincount(parent_ei[1], minlength=n_processes).float()

    # Process executable affiliation degrees
    executable_link_count = torch.bincount(executes_ei[0], minlength=n_processes).float()
    executable_degree = torch.bincount(executes_ei[1], minlength=n_executables).float()

    # Accumulate log popularity of each process's executable
    linked_executable_log_degree = torch.zeros(n_processes, dtype=torch.float)
    linked_executable_log_degree.index_add_(0, executes_ei[0], torch.log1p(executable_degree[executes_ei[1]]))
    mean_linked_executable_log_degree = linked_executable_log_degree / executable_link_count.clamp_min(1.0)

    features = pd.DataFrame({
        "parent_in_degree": parent_in_degree.numpy(),
        "parent_out_degree": parent_out_degree.numpy(),
        "has_children": (
            parent_out_degree > 0
        ).float().numpy(),
        "executable_link_count": executable_link_count.numpy(),
        "mean_linked_executable_log_degree": (
            mean_linked_executable_log_degree.numpy()
        ),
    })

    assert len(features) == n_processes
    assert np.isfinite(features.to_numpy()).all()

    return features
    
def pool_structural_features_to_sessions(process_features, process_session_ids, session_table):
    process_features = process_features.copy()
    process_features["session_id"] = np.asarray(process_session_ids, dtype=int)

    expected_session_ids = np.array(session_table["session_id"])

    session_features = (
        process_features
        .groupby("session_id", sort=True)
        .mean()
        .reindex(expected_session_ids)
    )

    session_sizes = np.bincount(np.array(process_session_ids), minlength=len(session_table))
    session_features["log_session_size"] = np.log1p(session_sizes)

    session_features = session_features.reset_index().rename(columns={"index": "session_id"})

    assert np.array_equal(
        np.array(session_features["session_id"]),
        expected_session_ids,
    )

    assert len(session_features) == len(session_table)
    assert session_features.isna().sum().sum() == 0
    
    return session_features

In [55]:
m1_process_features = build_process_structural_features(
    graph_data
)

m1_session_features = pool_structural_features_to_sessions(
    m1_process_features,
    graph_data["process"].session_id.cpu().numpy(),
    session_table,
)

m1_session_features

,session_id,parent_in_degree,parent_out_degree,has_children,executable_link_count,mean_linked_executable_log_degree,log_session_size
0,0,0.000000,0.000000,0.000000,1.000000,10.777127,5.003946
1,1,0.013699,0.013699,0.013699,0.986301,10.535172,4.990433
2,2,0.000000,0.000000,0.000000,1.000000,10.807639,4.897840
3,3,0.006494,0.006494,0.006494,1.000000,10.881991,5.043425
4,4,0.007143,0.007143,0.007143,1.000000,10.520494,4.948760
...,...,...,...,...,...,...,...
1452,1452,0.000000,0.000000,0.000000,1.000000,4.727388,0.693147
1453,1453,0.000000,0.000000,0.000000,1.000000,6.212606,0.693147
1454,1454,0.500000,0.500000,0.500000,0.500000,2.495216,1.098612
1455,1455,0.000000,0.000000,0.000000,1.000000,6.212606,0.693147


In [56]:
m1_feature_columns = [
    column
    for column in m1_session_features.columns
    if column != "session_id"
]

m1_scaler = StandardScaler()

X_m1 = m1_scaler.fit_transform(
    m1_session_features[m1_feature_columns]
)

m1_scores = knn_distance_anomaly_scores(
    X_m1,
    k=K,
)

assert_method_alignment(
    session_table,
    m1_session_features["session_id"].to_numpy(),
    m1_scores,
)

y_true = (
    session_table["label"]
    .map({"benign": 0, "malicious": 1})
    .to_numpy()
)

m1_result = evaluate_precomputed_scores(
    y_true,
    m1_scores,
)

m1_summary = metric_summary_from_method_results({
    "typed_structural_stats_knn": m1_result,
})

m1_summary

,method,average_precision,reviews_to_first_malicious,random_expected_reviews_to_first,found_at_25,recall_at_25,precision_at_25,found_at_50,recall_at_50,precision_at_50,found_at_100,recall_at_100,precision_at_100,found_at_250,recall_at_250,precision_at_250
0,typed_structural_stats_knn,0.025201,7,72.9,2,0.105263,0.08,2,0.105263,0.04,2,0.105263,0.02,4,0.210526,0.016


In [57]:
m1_neighbor_diagnostic = (
    malicious_neighbor_purity_diagnostic(
        X_m1,
        session_table,
        k=K,
    )
)

m1_neighbor_diagnostic

,diagnostic,k,n_sessions,n_malicious_sessions,base_malicious_rate_excluding_self,malicious_neighbor_purity_at_k,median_malicious_neighbor_purity_at_k,malicious_sessions_with_any_malicious_neighbor_at_k,enrichment_vs_random
0,malicious_neighbor_purity_at_k,15,1457,19,0.012363,0.259649,0.333333,1.0,21.002729


In [58]:
m1_standardized_features = pd.DataFrame(
    X_m1,
    columns=m1_feature_columns,
)

m1_standardized_features.insert(
    0,
    "session_id",
    m1_session_features["session_id"].to_numpy(),
)

m1_standardized_features["m1_score"] = m1_scores

top_m1_sessions = (
    m1_standardized_features
    .sort_values("m1_score", ascending=False)
    .head(10)
)

top_m1_sessions

,session_id,parent_in_degree,parent_out_degree,has_children,executable_link_count,mean_linked_executable_log_degree,log_session_size,m1_score
1294,1294,2.851676,32.421511,1.640583,0.195209,-0.789204,-0.367613,30.701964
513,513,2.731709,19.212284,2.479922,0.195209,-0.885743,-0.559302,18.473123
1362,1362,2.829182,4.924442,1.797959,0.195209,-0.807305,-0.409833,5.267289
1409,1409,0.489814,0.089835,1.797959,-2.609130,-2.397150,-0.954189,2.939845
1341,1341,1.016172,0.179549,3.025492,0.195209,-1.643476,-0.850320,2.754285
1321,1321,1.016172,0.179549,3.025492,0.195209,-1.643476,-0.850320,2.754285
1402,1402,-0.387450,0.538407,7.935624,0.195209,-1.887629,-1.476202,2.502541
1403,1403,-0.387450,0.538407,7.935624,0.195209,-1.887629,-1.476202,2.502541
1389,1389,-0.387450,0.538407,7.935624,0.195209,-1.887629,-1.476202,2.502541
1278,1278,-0.387450,0.538407,7.935624,0.195209,-1.887629,-1.476202,2.502541


In [59]:
top_m1_sessions[['parent_in_degree','parent_out_degree','has_children','mean_linked_executable_log_degree']].corrwith(top_m1_sessions['log_session_size'])

# and specifically for the headline example:
top_m1_sessions.loc[top_m1_sessions['session_id'] == 1294, 'log_session_size']

1294   -0.367613
Name: log_session_size, dtype: float64

#### M2 - Node2Vec

In [60]:
from torch_geometric.nn import Node2Vec
from torch_geometric.utils import coalesce

def build_node2vec_projection(graph_data):
    n_processes = graph_data["process"].num_nodes
    n_executables = graph_data["executable"].num_nodes
    n_nodes = n_processes + n_executables

    parent_ei = graph_data[
        "process", "parent_of", "process"
    ].edge_index.cpu()

    executes_ei = graph_data[
        "process", "executes", "executable"
    ].edge_index.cpu()

    # Offset executable indices into the homogeneous node space.
    executes_global = torch.stack([
        executes_ei[0],
        executes_ei[1] + n_processes,
    ])

    # Association-walk semantics: traverse both directions.
    edge_index = torch.cat([
        parent_ei,
        parent_ei.flip(0),
        executes_global,
        executes_global.flip(0),
    ], dim=1)

    edge_index = coalesce(
        edge_index,
        num_nodes=n_nodes,
    )

    degree = torch.bincount(
        edge_index[0],
        minlength=n_nodes,
    )

    projection_summary = pd.DataFrame([{
        "nodes": n_nodes,
        "process_nodes": n_processes,
        "executable_nodes": n_executables,
        "directed_walk_edges": edge_index.shape[1],
        "undirected_relations": edge_index.shape[1] // 2,
        "isolates": int((degree == 0).sum()),
    }])

    return edge_index, degree, projection_summary

In [61]:
node2vec_edge_index, node2vec_degree, node2vec_projection_summary = (
    build_node2vec_projection(graph_data)
)

node2vec_projection_summary

,nodes,process_nodes,executable_nodes,directed_walk_edges,undirected_relations,isolates
0,100053,100000,53,204336,102168,0


In [62]:
NODE2VEC_CONFIG = {
    "embedding_dim": 48,
    "walk_length": 10,
    "context_size": 5,
    "walks_per_node": 2,
    "p": 1.0,
    "q": 1.0,
    "num_negative_samples": 1,
    "epochs": 8,
    "batch_size": 1024,
    "learning_rate": 0.01,
}


def train_node2vec(
    edge_index,
    num_nodes,
    seed,
    config=NODE2VEC_CONFIG,
):
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model = Node2Vec(
        edge_index=edge_index,
        embedding_dim=config["embedding_dim"],
        walk_length=config["walk_length"],
        context_size=config["context_size"],
        walks_per_node=config["walks_per_node"],
        p=config["p"],
        q=config["q"],
        num_negative_samples=config[
            "num_negative_samples"
        ],
        num_nodes=num_nodes,
        sparse=True,
    ).to(device)

    loader = model.loader(
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=0,
    )

    optimizer = torch.optim.SparseAdam(
        model.parameters(),
        lr=config["learning_rate"],
    )

    epoch_losses = []

    for epoch in range(config["epochs"]):
        model.train()
        total_loss = 0.0
        batches = 0

        for positive_walks, negative_walks in loader:
            optimizer.zero_grad()

            loss = model.loss(
                positive_walks.to(device),
                negative_walks.to(device),
            )

            loss.backward()
            optimizer.step()

            total_loss += loss.detach().item()
            batches += 1

        mean_loss = total_loss / max(batches, 1)
        epoch_losses.append(mean_loss)

        print(
            f"seed={seed} "
            f"epoch={epoch + 1:02d} "
            f"loss={mean_loss:.4f}"
        )

    embeddings = model().detach().cpu()

    return embeddings, epoch_losses

In [63]:
def mean_pool_process_embeddings(
    process_embeddings,
    process_session_ids,
    n_sessions,
):
    """Mean-pool processes and append the fixed log-session-size channel.

    The explicit size channel is required by the shared representation
    contract and is applied identically to M2 and every M3 control.
    """
    process_session_ids = torch.as_tensor(
        process_session_ids,
        dtype=torch.long,
    )

    assert process_embeddings.ndim == 2
    assert process_session_ids.ndim == 1
    assert len(process_embeddings) == len(process_session_ids)
    assert int(process_session_ids.min()) >= 0
    assert int(process_session_ids.max()) < n_sessions

    pooled = torch.zeros(
        (n_sessions, process_embeddings.shape[1]),
        dtype=process_embeddings.dtype,
    )

    counts = torch.zeros(
        n_sessions,
        dtype=process_embeddings.dtype,
    )

    pooled.index_add_(
        0,
        process_session_ids,
        process_embeddings,
    )

    counts.index_add_(
        0,
        process_session_ids,
        torch.ones(
            len(process_session_ids),
            dtype=process_embeddings.dtype,
        ),
    )

    assert torch.all(counts > 0)

    mean_pooled = pooled / counts.unsqueeze(1)
    log_session_size = torch.log1p(counts).unsqueeze(1)
    session_representation = torch.cat([
        mean_pooled,
        log_session_size,
    ], dim=1)

    assert session_representation.shape == (
        n_sessions,
        process_embeddings.shape[1] + 1,
    )
    assert torch.isfinite(session_representation).all()

    return session_representation


def assert_graph_session_representation(
    session_representation,
    base_dimension,
):
    """Verify alignment and the shared explicit size channel."""
    assert session_representation.shape == (
        len(session_table),
        base_dimension + 1,
    )

    expected_log_session_size = torch.as_tensor(
        m1_session_features["log_session_size"].to_numpy(),
        dtype=session_representation.dtype,
    )
    assert torch.allclose(
        session_representation[:, -1].cpu(),
        expected_log_session_size,
        atol=1e-6,
        rtol=1e-6,
    )
    assert torch.isfinite(session_representation).all()

In [64]:
m2_seed_summaries = []
m2_scores_by_seed = {}
m2_embeddings_by_seed = {}
m2_losses_by_seed = {}
m2_diagnostics_by_seed = {}

n_processes = graph_data["process"].num_nodes
n_node2vec_nodes = (
    graph_data["process"].num_nodes
    + graph_data["executable"].num_nodes
)

isolated_nodes = node2vec_degree == 0

for seed in EXPERIMENT_SEEDS:
    node_embeddings, losses = train_node2vec(
        node2vec_edge_index,
        num_nodes=n_node2vec_nodes,
        seed=seed,
    )

    # Explicit isolate handling.
    node_embeddings[isolated_nodes] = 0.0

    process_embeddings = node_embeddings[:n_processes]

    session_embeddings = mean_pool_process_embeddings(
        process_embeddings,
        graph_data["process"].session_id.cpu(),
        n_sessions=len(session_table),
    )
    assert_graph_session_representation(
        session_embeddings,
        base_dimension=NODE2VEC_CONFIG["embedding_dim"],
    )

    X_m2 = StandardScaler().fit_transform(
        session_embeddings.numpy()
    )

    m2_scores = knn_distance_anomaly_scores(
        X_m2,
        k=K,
    )

    assert_method_alignment(
        session_table,
        session_table["session_id"].to_numpy(),
        m2_scores,
    )

    m2_result = evaluate_precomputed_scores(
        y_true,
        m2_scores,
    )

    seed_summary = metric_summary_from_method_results({
        "node2vec_session_knn": m2_result,
    })

    seed_summary.insert(0, "seed", seed)
    m2_seed_summaries.append(seed_summary)

    m2_scores_by_seed[seed] = m2_scores
    m2_embeddings_by_seed[seed] = session_embeddings
    m2_losses_by_seed[seed] = losses

    m2_diagnostics_by_seed[seed] = (
        malicious_neighbor_purity_diagnostic(
            X_m2,
            session_table,
            k=K,
        )
    )

m2_seed_summary = pd.concat(
    m2_seed_summaries,
    ignore_index=True,
)

m2_seed_summary

seed=42 epoch=01 loss=4.4625
seed=42 epoch=02 loss=3.4041
seed=42 epoch=03 loss=2.8237
seed=42 epoch=04 loss=2.3839
seed=42 epoch=05 loss=2.0429
seed=42 epoch=06 loss=1.7809
seed=42 epoch=07 loss=1.5827
seed=42 epoch=08 loss=1.4281
seed=43 epoch=01 loss=4.4587
seed=43 epoch=02 loss=3.4069
seed=43 epoch=03 loss=2.8227
seed=43 epoch=04 loss=2.3783
seed=43 epoch=05 loss=2.0390
seed=43 epoch=06 loss=1.7812
seed=43 epoch=07 loss=1.5837
seed=43 epoch=08 loss=1.4259
seed=44 epoch=01 loss=4.3472
seed=44 epoch=02 loss=3.3990
seed=44 epoch=03 loss=2.8507
seed=44 epoch=04 loss=2.3930
seed=44 epoch=05 loss=2.0369
seed=44 epoch=06 loss=1.7719
seed=44 epoch=07 loss=1.5725
seed=44 epoch=08 loss=1.4176


,seed,method,average_precision,reviews_to_first_malicious,random_expected_reviews_to_first,found_at_25,recall_at_25,precision_at_25,found_at_50,recall_at_50,precision_at_50,found_at_100,recall_at_100,precision_at_100,found_at_250,recall_at_250,precision_at_250
0,42,node2vec_session_knn,0.045818,47,72.9,0,0.000000,0.00,1,0.052632,0.02,2,0.105263,0.02,15,0.789474,0.060
1,43,node2vec_session_knn,0.097340,1,72.9,1,0.052632,0.04,1,0.052632,0.02,3,0.157895,0.03,15,0.789474,0.060
2,44,node2vec_session_knn,0.056985,7,72.9,1,0.052632,0.04,2,0.105263,0.04,3,0.157895,0.03,16,0.842105,0.064


In [65]:
m2_metric_columns = [
    column
    for column in m2_seed_summary.columns
    if column not in {"seed", "method"}
]

m2_aggregate_summary = (
    m2_seed_summary[m2_metric_columns]
    .agg(["mean", "std"])
    .T
)

m2_aggregate_summary

,mean,std
average_precision,0.066715,0.027104
reviews_to_first_malicious,18.333333,25.006666
random_expected_reviews_to_first,72.900000,0.000000
found_at_25,0.666667,0.577350
recall_at_25,0.035088,0.030387
precision_at_25,0.026667,0.023094
found_at_50,1.333333,0.577350
recall_at_50,0.070175,0.030387
precision_at_50,0.026667,0.011547
found_at_100,2.666667,0.577350


In [66]:
m2_diagnostic_summary = pd.concat([
    diagnostic.assign(seed=seed)
    for seed, diagnostic
    in m2_diagnostics_by_seed.items()
], ignore_index=True)

m2_diagnostic_summary

,diagnostic,k,n_sessions,n_malicious_sessions,base_malicious_rate_excluding_self,malicious_neighbor_purity_at_k,median_malicious_neighbor_purity_at_k,malicious_sessions_with_any_malicious_neighbor_at_k,enrichment_vs_random,seed
0,malicious_neighbor_purity_at_k,15,1457,19,0.012363,0.010526,0.0,0.157895,0.851462,42
1,malicious_neighbor_purity_at_k,15,1457,19,0.012363,0.003509,0.0,0.052632,0.283821,43
2,malicious_neighbor_purity_at_k,15,1457,19,0.012363,0.003509,0.0,0.052632,0.283821,44


#### M3 - Relational GraphSAGE

In [67]:
from torch_geometric.utils import negative_sampling


M3_CONFIG = {
    "hidden_dim": 48,
    "output_dim": 48,
    "epochs": 8,
    "learning_rate": 0.001,
    "dropout": 0.1,
    "max_positive_edges_per_relation": 5_000,
}


class RelationalGraphSAGE(nn.Module):
    def __init__(
        self,
        hidden_dim=48,
        output_dim=48,
        dropout=0.1,
    ):
        super().__init__()

        edge_types = [
            ("process", "parent_of", "process"),
            ("process", "child_of", "process"),
            ("process", "executes", "executable"),
            ("executable", "executed_by", "process"),
        ]

        self.conv1 = HeteroConv(
            {
                edge_type: SAGEConv(
                    (-1, -1),
                    hidden_dim,
                )
                for edge_type in edge_types
            },
            aggr="sum",
        )

        self.conv2 = HeteroConv(
            {
                edge_type: SAGEConv(
                    (-1, -1),
                    output_dim,
                )
                for edge_type in edge_types
            },
            aggr="sum",
        )

        self.dropout = dropout

    def forward(self, x_dict, edge_index_dict):
        hidden = self.conv1(
            x_dict,
            edge_index_dict,
        )

        hidden = {
            node_type: F.dropout(
                F.relu(node_features),
                p=self.dropout,
                training=self.training,
            )
            for node_type, node_features in hidden.items()
        }

        return self.conv2(
            hidden,
            edge_index_dict,
        )


class TypedLinkDecoder(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()

        # Bilinear decoders preserve source/target direction.
        self.decoders = nn.ModuleDict({
            "parent_of": nn.Bilinear(
                embedding_dim,
                embedding_dim,
                1,
            ),
            "executes": nn.Bilinear(
                embedding_dim,
                embedding_dim,
                1,
            ),
        })

    def forward(
        self,
        relation_name,
        source_embeddings,
        target_embeddings,
        edge_index,
    ):
        source = F.normalize(
            source_embeddings[edge_index[0]],
            p=2,
            dim=-1,
        )
        target = F.normalize(
            target_embeddings[edge_index[1]],
            p=2,
            dim=-1,
        )

        return self.decoders[relation_name](
            source,
            target,
        ).squeeze(-1)

In [68]:
m3_process_scaler = StandardScaler()

m3_process_x = torch.tensor(
    m3_process_scaler.fit_transform(
        graph_data["process"].x.cpu().numpy()
    ),
    dtype=torch.float,
)

m3_executable_x = (
    graph_data["executable"].x
    .detach()
    .cpu()
    .float()
)

m3_x_dict = {
    "process": m3_process_x,
    "executable": m3_executable_x,
}

m3_edge_index_dict = {
    edge_type: edge_index.cpu()
    for edge_type, edge_index
    in graph_data.edge_index_dict.items()
}

assert torch.isfinite(m3_process_x).all()
assert torch.isfinite(m3_executable_x).all()

In [69]:
def train_relational_graphsage(
    x_dict,
    edge_index_dict,
    seed,
    config=M3_CONFIG,
):
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    x_dict_device = {
        node_type: features.to(device)
        for node_type, features in x_dict.items()
    }

    edge_index_dict_device = {
        edge_type: edge_index.to(device)
        for edge_type, edge_index
        in edge_index_dict.items()
    }

    encoder = RelationalGraphSAGE(
        hidden_dim=config["hidden_dim"],
        output_dim=config["output_dim"],
        dropout=config["dropout"],
    ).to(device)

    decoder = TypedLinkDecoder(
        embedding_dim=config["output_dim"]
    ).to(device)

    # Materialize lazy SAGEConv dimensions.
    encoder.eval()

    with torch.no_grad():
        random_embeddings = encoder(
            x_dict_device,
            edge_index_dict_device,
        )

        random_process_embeddings = (
            random_embeddings["process"]
            .detach()
            .cpu()
        )

    parameters = (
        list(encoder.parameters())
        + list(decoder.parameters())
    )

    optimizer = torch.optim.Adam(
        parameters,
        lr=config["learning_rate"],
    )

    relation_specs = {
        "parent_of": {
            "edge_type": (
                "process",
                "parent_of",
                "process",
            ),
            "source_type": "process",
            "target_type": "process",
            "num_nodes": (
                x_dict["process"].shape[0],
                x_dict["process"].shape[0],
            ),
        },
        "executes": {
            "edge_type": (
                "process",
                "executes",
                "executable",
            ),
            "source_type": "process",
            "target_type": "executable",
            "num_nodes": (
                x_dict["process"].shape[0],
                x_dict["executable"].shape[0],
            ),
        },
    }

    epoch_losses = []

    for epoch in range(config["epochs"]):
        encoder.train()
        decoder.train()
        optimizer.zero_grad()

        embeddings = encoder(
            x_dict_device,
            edge_index_dict_device,
        )

        relation_losses = []

        for relation_name, spec in relation_specs.items():
            full_positive_edges = edge_index_dict[
                spec["edge_type"]
            ]

            n_positive = min(
                full_positive_edges.shape[1],
                config[
                    "max_positive_edges_per_relation"
                ],
            )

            permutation = torch.randperm(
                full_positive_edges.shape[1]
            )[:n_positive]

            positive_edges = (
                full_positive_edges[:, permutation]
                .to(device)
            )

            negative_edges = negative_sampling(
                edge_index=full_positive_edges,
                num_nodes=spec["num_nodes"],
                num_neg_samples=n_positive,
                method="sparse",
            ).to(device)

            positive_logits = decoder(
                relation_name,
                embeddings[spec["source_type"]],
                embeddings[spec["target_type"]],
                positive_edges,
            )

            negative_logits = decoder(
                relation_name,
                embeddings[spec["source_type"]],
                embeddings[spec["target_type"]],
                negative_edges,
            )

            positive_loss = (
                F.binary_cross_entropy_with_logits(
                    positive_logits,
                    torch.ones_like(positive_logits),
                )
            )

            negative_loss = (
                F.binary_cross_entropy_with_logits(
                    negative_logits,
                    torch.zeros_like(negative_logits),
                )
            )

            relation_losses.append(
                positive_loss + negative_loss
            )

        loss = torch.stack(relation_losses).mean()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(parameters, max_norm=1.0)
        optimizer.step()

        assert torch.isfinite(loss).all()

        epoch_loss = loss.detach().item()
        epoch_losses.append(epoch_loss)

        print(
            f"seed={seed} "
            f"epoch={epoch + 1:02d} "
            f"loss={epoch_loss:.4f}"
        )

    encoder.eval()

    with torch.no_grad():
        trained_embeddings = encoder(
            x_dict_device,
            edge_index_dict_device,
        )

        trained_process_embeddings = (
            trained_embeddings["process"]
            .detach()
            .cpu()
        )

    return {
        "random_init": random_process_embeddings,
        "trained": trained_process_embeddings,
        "losses": epoch_losses,
    }

In [70]:
m3_seed_summaries = []
m3_scores_by_seed = {}
m3_embeddings_by_seed = {}
m3_losses_by_seed = {}
m3_diagnostics_by_seed = {}

process_session_ids = (
    graph_data["process"]
    .session_id
    .cpu()
)

for seed in EXPERIMENT_SEEDS:
    run = train_relational_graphsage(
        m3_x_dict,
        m3_edge_index_dict,
        seed=seed,
    )

    m3_losses_by_seed[seed] = run["losses"]

    for variant in ("random_init", "trained"):
        process_embeddings = run[variant]

        session_embeddings = mean_pool_process_embeddings(
            process_embeddings,
            process_session_ids,
            n_sessions=len(session_table),
        )
        assert_graph_session_representation(
            session_embeddings,
            base_dimension=M3_CONFIG["output_dim"],
        )

        X_m3 = StandardScaler().fit_transform(
            session_embeddings.numpy()
        )

        scores = knn_distance_anomaly_scores(
            X_m3,
            k=K,
        )

        assert_method_alignment(
            session_table,
            session_table["session_id"].to_numpy(),
            scores,
        )

        method_name = (
            f"relational_graphsage_{variant}"
        )

        result = evaluate_precomputed_scores(
            y_true,
            scores,
        )

        summary = metric_summary_from_method_results({
            method_name: result,
        })

        summary.insert(0, "seed", seed)
        m3_seed_summaries.append(summary)

        key = (seed, variant)

        m3_scores_by_seed[key] = scores
        m3_embeddings_by_seed[key] = (
            session_embeddings
        )

        m3_diagnostics_by_seed[key] = (
            malicious_neighbor_purity_diagnostic(
                X_m3,
                session_table,
                k=K,
            )
        )

m3_seed_summary = pd.concat(
    m3_seed_summaries,
    ignore_index=True,
)

m3_seed_summary

seed=42 epoch=01 loss=1.3888
seed=42 epoch=02 loss=1.3400
seed=42 epoch=03 loss=1.3110
seed=42 epoch=04 loss=1.2786
seed=42 epoch=05 loss=1.2542
seed=42 epoch=06 loss=1.2363
seed=42 epoch=07 loss=1.2107
seed=42 epoch=08 loss=1.1856
seed=43 epoch=01 loss=1.3947
seed=43 epoch=02 loss=1.3420
seed=43 epoch=03 loss=1.3036
seed=43 epoch=04 loss=1.2950
seed=43 epoch=05 loss=1.2578
seed=43 epoch=06 loss=1.2451
seed=43 epoch=07 loss=1.2217
seed=43 epoch=08 loss=1.1926
seed=44 epoch=01 loss=1.4056
seed=44 epoch=02 loss=1.3736
seed=44 epoch=03 loss=1.3427
seed=44 epoch=04 loss=1.3078
seed=44 epoch=05 loss=1.2958
seed=44 epoch=06 loss=1.2623
seed=44 epoch=07 loss=1.2372
seed=44 epoch=08 loss=1.2105


,seed,method,average_precision,reviews_to_first_malicious,random_expected_reviews_to_first,found_at_25,recall_at_25,precision_at_25,found_at_50,recall_at_50,precision_at_50,found_at_100,recall_at_100,precision_at_100,found_at_250,recall_at_250,precision_at_250
0,42,relational_graphsage_random_init,0.020795,87,72.9,0,0.0,0.0,0,0.000000,0.00,1,0.052632,0.01,2,0.105263,0.008
1,42,relational_graphsage_trained,0.017811,99,72.9,0,0.0,0.0,0,0.000000,0.00,1,0.052632,0.01,2,0.105263,0.008
2,43,relational_graphsage_random_init,0.021522,70,72.9,0,0.0,0.0,0,0.000000,0.00,2,0.105263,0.02,3,0.157895,0.012
3,43,relational_graphsage_trained,0.019658,63,72.9,0,0.0,0.0,0,0.000000,0.00,2,0.105263,0.02,2,0.105263,0.008
4,44,relational_graphsage_random_init,0.018251,62,72.9,0,0.0,0.0,0,0.000000,0.00,2,0.105263,0.02,3,0.157895,0.012
5,44,relational_graphsage_trained,0.021041,50,72.9,0,0.0,0.0,1,0.052632,0.02,2,0.105263,0.02,4,0.210526,0.016


In [71]:
m3_metric_columns = [
    column
    for column in m3_seed_summary.columns
    if column not in {"seed", "method"}
]

m3_aggregate_summary = (
    m3_seed_summary
    .groupby("method")[m3_metric_columns]
    .agg(["mean", "std"])
)

m3_aggregate_summary

average_precision            \
                                              mean       std   
method                                                         
relational_graphsage_random_init          0.020189  0.001718   
relational_graphsage_trained              0.019503  0.001621   

                                 reviews_to_first_malicious             \
                                                       mean        std   
method                                                                   
relational_graphsage_random_init                  73.000000  12.767145   
relational_graphsage_trained                      70.666667  25.383722   

                                 random_expected_reviews_to_first       \
                                                             mean  std   
method                                                                   
relational_graphsage_random_init                             72.9  0.0   
relational_graphsage_trained                                 72.9  0.0   

                                 found_at_25      recall_at_25       ...  \
                                        mean  std         mean  std  ...   
method                                                               ...   
relational_graphsage_random_init         0.0  0.0          0.0  0.0  ...   
relational_graphsage_trained             0.0  0.0          0.0  0.0  ...   

                                 recall_at_100           precision_at_100  \
                                          mean       std             mean   
method                                                                      
relational_graphsage_random_init      0.087719  0.030387         0.016667   
relational_graphsage_trained          0.087719  0.030387         0.016667   

                                           found_at_250            \
                                       std         mean       std   
method                                                              
relational_graphsage_random_init  0.005774     2.666667  0.577350   
relational_graphsage_trained      0.005774     2.666667  1.154701   

                                 recall_at_250           precision_at_250  \
                                          mean       std             mean   
method                                                                      
relational_graphsage_random_init      0.140351  0.030387         0.010667   
relational_graphsage_trained          0.140351  0.060774         0.010667   

                                            
                                       std  
method                                      
relational_graphsage_random_init  0.002309  
relational_graphsage_trained      0.004619  

[2 rows x 30 columns]

In [72]:
# M3 validation: masked typed-edge reconstruction.
# Positive targets are removed in both directions before message passing,
# so the encoder cannot directly consume the edges it is asked to predict.
MASKED_M3_CONFIG = {
    **M3_CONFIG,
    "mask_fraction": 0.20,
}


def train_masked_relational_graphsage(
    x_dict,
    edge_index_dict,
    seed,
    config=MASKED_M3_CONFIG,
):
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    relation_specs = {
        "parent_of": {
            "edge_type": ("process", "parent_of", "process"),
            "reverse_edge_type": ("process", "child_of", "process"),
            "source_type": "process",
            "target_type": "process",
            "num_nodes": (
                x_dict["process"].shape[0],
                x_dict["process"].shape[0],
            ),
        },
        "executes": {
            "edge_type": ("process", "executes", "executable"),
            "reverse_edge_type": ("executable", "executed_by", "process"),
            "source_type": "process",
            "target_type": "executable",
            "num_nodes": (
                x_dict["process"].shape[0],
                x_dict["executable"].shape[0],
            ),
        },
    }

    # Use a fixed, seed-specific split. The full positive set remains available
    # only to prevent negative sampling from creating false negatives.
    split_generator = torch.Generator().manual_seed(seed)
    message_edge_index_dict = dict(edge_index_dict)
    reconstruction_edges = {}
    masked_edge_counts = {}

    for relation_name, spec in relation_specs.items():
        full_edges = edge_index_dict[spec["edge_type"]].cpu()
        n_edges = full_edges.shape[1]
        assert n_edges > 1

        n_masked = min(
            max(1, int(round(config["mask_fraction"] * n_edges))),
            n_edges - 1,
        )
        permutation = torch.randperm(
            n_edges,
            generator=split_generator,
        )
        masked_edges = full_edges[:, permutation[:n_masked]]
        retained_edges = full_edges[:, permutation[n_masked:]]

        reconstruction_edges[relation_name] = masked_edges
        masked_edge_counts[relation_name] = n_masked
        message_edge_index_dict[spec["edge_type"]] = retained_edges
        message_edge_index_dict[spec["reverse_edge_type"]] = (
            retained_edges.flip(0)
        )

    x_dict_device = {
        node_type: features.to(device)
        for node_type, features in x_dict.items()
    }
    message_edge_index_dict_device = {
        edge_type: edge_index.to(device)
        for edge_type, edge_index in message_edge_index_dict.items()
    }
    full_edge_index_dict_device = {
        edge_type: edge_index.to(device)
        for edge_type, edge_index in edge_index_dict.items()
    }

    encoder = RelationalGraphSAGE(
        hidden_dim=config["hidden_dim"],
        output_dim=config["output_dim"],
        dropout=config["dropout"],
    ).to(device)
    decoder = TypedLinkDecoder(
        embedding_dim=config["output_dim"]
    ).to(device)

    # Both controls use the restored full graph at inference. Only the
    # reconstruction-training forward passes use the masked graph.
    encoder.eval()
    with torch.no_grad():
        random_embeddings = encoder(
            x_dict_device,
            full_edge_index_dict_device,
        )
        random_process_embeddings = (
            random_embeddings["process"].detach().cpu()
        )

    parameters = list(encoder.parameters()) + list(decoder.parameters())
    optimizer = torch.optim.Adam(
        parameters,
        lr=config["learning_rate"],
    )
    epoch_losses = []

    for epoch in range(config["epochs"]):
        encoder.train()
        decoder.train()
        optimizer.zero_grad()

        embeddings = encoder(
            x_dict_device,
            message_edge_index_dict_device,
        )
        relation_losses = []

        for relation_name, spec in relation_specs.items():
            candidate_positive_edges = reconstruction_edges[relation_name]
            n_positive = min(
                candidate_positive_edges.shape[1],
                config["max_positive_edges_per_relation"],
            )
            permutation = torch.randperm(
                candidate_positive_edges.shape[1]
            )[:n_positive]
            positive_edges = (
                candidate_positive_edges[:, permutation].to(device)
            )

            # Sample against all real edges, including masked targets.
            negative_edges = negative_sampling(
                edge_index=edge_index_dict[spec["edge_type"]],
                num_nodes=spec["num_nodes"],
                num_neg_samples=n_positive,
                method="sparse",
            ).to(device)

            positive_logits = decoder(
                relation_name,
                embeddings[spec["source_type"]],
                embeddings[spec["target_type"]],
                positive_edges,
            )
            negative_logits = decoder(
                relation_name,
                embeddings[spec["source_type"]],
                embeddings[spec["target_type"]],
                negative_edges,
            )

            relation_losses.append(
                F.binary_cross_entropy_with_logits(
                    positive_logits,
                    torch.ones_like(positive_logits),
                )
                + F.binary_cross_entropy_with_logits(
                    negative_logits,
                    torch.zeros_like(negative_logits),
                )
            )

        # Equal weighting prevents the large executes relation from
        # overwhelming the much smaller lineage relation.
        loss = torch.stack(relation_losses).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(parameters, max_norm=1.0)
        optimizer.step()
        assert torch.isfinite(loss)

        epoch_loss = loss.detach().item()
        epoch_losses.append(epoch_loss)
        print(
            f"masked seed={seed} epoch={epoch + 1:02d} "
            f"loss={epoch_loss:.4f}"
        )

    encoder.eval()
    with torch.no_grad():
        trained_embeddings = encoder(
            x_dict_device,
            full_edge_index_dict_device,
        )
        trained_process_embeddings = (
            trained_embeddings["process"].detach().cpu()
        )

    return {
        "random_init": random_process_embeddings,
        "trained": trained_process_embeddings,
        "losses": epoch_losses,
        "masked_edge_counts": masked_edge_counts,
    }


masked_m3_seed_summaries = []
masked_m3_scores_by_seed = {}
masked_m3_losses_by_seed = {}
masked_m3_edge_counts_by_seed = {}

for seed in EXPERIMENT_SEEDS:
    run = train_masked_relational_graphsage(
        m3_x_dict,
        m3_edge_index_dict,
        seed=seed,
    )
    masked_m3_losses_by_seed[seed] = run["losses"]
    masked_m3_edge_counts_by_seed[seed] = run["masked_edge_counts"]

    for variant in ("random_init", "trained"):
        session_embeddings = mean_pool_process_embeddings(
            run[variant],
            process_session_ids,
            n_sessions=len(session_table),
        )
        assert_graph_session_representation(
            session_embeddings,
            base_dimension=MASKED_M3_CONFIG["output_dim"],
        )
        X_masked_m3 = StandardScaler().fit_transform(
            session_embeddings.numpy()
        )
        scores = knn_distance_anomaly_scores(X_masked_m3, k=K)
        assert_method_alignment(
            session_table,
            session_table["session_id"].to_numpy(),
            scores,
        )

        method_name = f"relational_graphsage_masked_{variant}"
        result = evaluate_precomputed_scores(y_true, scores)
        summary = metric_summary_from_method_results({method_name: result})
        summary.insert(0, "seed", seed)
        masked_m3_seed_summaries.append(summary)
        masked_m3_scores_by_seed[(seed, variant)] = scores

masked_m3_seed_summary = pd.concat(
    masked_m3_seed_summaries,
    ignore_index=True,
)
masked_m3_metric_columns = [
    column
    for column in masked_m3_seed_summary.columns
    if column not in {"seed", "method"}
]
masked_m3_aggregate_summary = (
    masked_m3_seed_summary
    .groupby("method")[masked_m3_metric_columns]
    .agg(["mean", "std"])
)

display(pd.DataFrame.from_dict(
    masked_m3_edge_counts_by_seed,
    orient="index",
).rename_axis("seed"))
masked_m3_aggregate_summary

masked seed=42 epoch=01 loss=1.3792
masked seed=42 epoch=02 loss=1.3366
masked seed=42 epoch=03 loss=1.2997
masked seed=42 epoch=04 loss=1.2698
masked seed=42 epoch=05 loss=1.2481
masked seed=42 epoch=06 loss=1.2204
masked seed=42 epoch=07 loss=1.1980
masked seed=42 epoch=08 loss=1.1648
masked seed=43 epoch=01 loss=1.4009
masked seed=43 epoch=02 loss=1.3605
masked seed=43 epoch=03 loss=1.3221
masked seed=43 epoch=04 loss=1.2971
masked seed=43 epoch=05 loss=1.2650
masked seed=43 epoch=06 loss=1.2403
masked seed=43 epoch=07 loss=1.2104
masked seed=43 epoch=08 loss=1.1899
masked seed=44 epoch=01 loss=1.3783
masked seed=44 epoch=02 loss=1.3572
masked seed=44 epoch=03 loss=1.3192
masked seed=44 epoch=04 loss=1.2946
masked seed=44 epoch=05 loss=1.2660
masked seed=44 epoch=06 loss=1.2431
masked seed=44 epoch=07 loss=1.2100
masked seed=44 epoch=08 loss=1.1886


,parent_of,executes
seed,,
42,464,19970
43,464,19970
44,464,19970


average_precision            \
                                                     mean       std   
method                                                                
relational_graphsage_masked_random_init          0.020189  0.001718   
relational_graphsage_masked_trained              0.019657  0.001822   

                                        reviews_to_first_malicious             \
                                                              mean        std   
method                                                                          
relational_graphsage_masked_random_init                  73.000000  12.767145   
relational_graphsage_masked_trained                      80.666667  29.938827   

                                        random_expected_reviews_to_first       \
                                                                    mean  std   
method                                                                          
relational_graphsage_masked_random_init                             72.9  0.0   
relational_graphsage_masked_trained                                 72.9  0.0   

                                        found_at_25      recall_at_25       \
                                               mean  std         mean  std   
method                                                                       
relational_graphsage_masked_random_init         0.0  0.0          0.0  0.0   
relational_graphsage_masked_trained             0.0  0.0          0.0  0.0   

                                         ... recall_at_100            \
                                         ...          mean       std   
method                                   ...                           
relational_graphsage_masked_random_init  ...      0.087719  0.030387   
relational_graphsage_masked_trained      ...      0.070175  0.060774   

                                        precision_at_100            \
                                                    mean       std   
method                                                               
relational_graphsage_masked_random_init         0.016667  0.005774   
relational_graphsage_masked_trained             0.013333  0.011547   

                                        found_at_250           recall_at_250  \
                                                mean       std          mean   
method                                                                         
relational_graphsage_masked_random_init     2.666667  0.577350      0.140351   
relational_graphsage_masked_trained         3.666667  1.527525      0.192982   

                                                  precision_at_250            
                                              std             mean       std  
method                                                                        
relational_graphsage_masked_random_init  0.030387         0.010667  0.002309  
relational_graphsage_masked_trained      0.080396         0.014667  0.006110  

[2 rows x 30 columns]

## Pipeline Entry Point

In [78]:
def run_triage_methods(config):

    ### Prepare Data ###
    # Load the development slice of the data
    dev_df = load_acme_slice()
    slice_manifest = get_slice_manifest(dev_df)
    # slice_manifest.to_csv("../artifacts/descriptions/development_slice_manifest.csv", index=False)

    session_tables = {}
    output_scores = {}

    for method_name, method_func in config.items():
        if method_name == "random_score_baseline":
            ### Run Random Scoring Baseline ###
            sessionized_df, sessions = make_strawman_sessions(dev_df)
            session_tables[method_name] = build_raw_session_stats(sessionized_df)

            y_true = session_tables[method_name]["label"].map({"malicious": 1, "benign": 0}).to_numpy(dtype=int)
            
            random_scores = random_anomaly_scores(len(session_tables[method_name]), seed=SEED)
            session_tables[method_name]["random_score"] = random_scores

            output_scores[method_name] = evaluate_precomputed_scores(y_true, random_scores)
        
        elif method_name == "raw_session_baseline":
            ### Run Strawman Sessionization and Raw Session Stats ###

            # Create a session table with raw session statistics
            sessionized_df, sessions = method_func(dev_df)
            session_tables[method_name] = build_raw_session_stats(sessionized_df)

            # Get the feature columns for scoring, excluding session_id and label
            feature_columns = [c for c in session_tables[method_name].columns if c not in {"session_id", "label"}]

            # Score on the raw session stats using StandardScaler and kNN anomaly scores
            X = StandardScaler().fit_transform(session_tables[method_name][feature_columns])

            # Scoring

            # Get back a dictionary of triage metrics, including recall, percision, reviews to retrival, anomaly distance z-scores and representation diagnostics
            triage_metrics = triage_metric_eval(X, session_tables[method_name])
            output_scores[method_name] = triage_metrics


        elif method_name == "graph_session_baseline":
            ### Run Graph Sessionization ###
            graph_builder = method_func
            graph_builder.build(dev_df)
            sessions = graph_builder.sessions
            # session_tables[method_name] = graph_builder.session_stats(graph_builder, dev_df)

            X = None

            # Get back a dictionary of triage metrics, including recall, percision, reviews to retrival, anomaly distance z-scores and representation diagnostics
            triage_metrics = triage_metric_eval(X, session_tables[method_name], k=15)
            output_scores[method_name] = triage_metrics

    
    metric_summary = metric_summary_from_method_results(output_scores)

    results = {
        "slice_manifest": slice_manifest,
        "session_table": session_tables,
        "triage_metrics_per_method": output_scores,
        "metric_summary": metric_summary,
    }

    return dev_df, results


graph_builder = TelemetryGraphBuilder()
methods_config = {
    "random_score_baseline": None,
    "raw_session_baseline": make_strawman_sessions,
    # "graph_session_baseline": graph_builder,
}

dev_df, results = run_triage_methods(methods_config)

count    1457.000000
mean        3.016148
std         8.749429
min         0.539398
25%         1.070602
50%         1.603439
75%         2.398985
max       239.544023
Name: raw_session_stats_knn, dtype: float64

1294    239.544023
513     144.003089
506      65.312406
1362     56.404498
32       54.024441
161      53.749397
488      42.293852
558      40.640419
835      38.089992
12       35.424416
Name: raw_session_stats_knn, dtype: float64

,anomaly_distance_z_scores
0,-0.010503
1,0.028106
2,-0.124306
3,-0.117825
4,0.490602
...,...
1452,-0.026515
1453,-0.162031
1454,-0.072907
1455,-0.168537


,diagnostic,k,n_sessions,n_malicious_sessions,base_malicious_rate_excluding_self,malicious_neighbor_purity_at_k,median_malicious_neighbor_purity_at_k,malicious_sessions_with_any_malicious_neighbor_at_k,enrichment_vs_random
0,malicious_neighbor_purity_at_k,15,1457,19,0.012363,0.207018,0.266667,0.842105,16.745419


In [79]:
results['metric_summary']

,method,average_precision,reviews_to_first_malicious,random_expected_reviews_to_first,found_at_25,recall_at_25,precision_at_25,found_at_50,recall_at_50,precision_at_50,found_at_100,recall_at_100,precision_at_100,found_at_250,recall_at_250,precision_at_250
0,random_score_baseline,0.014803,47,72.9,0,0.0,0.0,1,0.052632,0.02,1,0.052632,0.01,3,0.157895,0.012
1,raw_session_baseline,0.016299,128,72.9,0,0.0,0.0,0,0.000000,0.00,0,0.000000,0.00,2,0.105263,0.008


In [80]:
results['triage_metrics_per_method']

{'random_score_baseline': {'average_precision': 0.01480310492496753,
  'reviews_to_first_malicious': 47,
  'random_expected_reviews_to_first': 72.9,
  'ranking_at_k_metrics': {'found_at_25': 0,
   'recall_at_25': 0.0,
   'precision_at_25': 0.0,
   'found_at_50': 1,
   'recall_at_50': 0.05263157894736842,
   'precision_at_50': 0.02,
   'found_at_100': 1,
   'recall_at_100': 0.05263157894736842,
   'precision_at_100': 0.01,
   'found_at_250': 3,
   'recall_at_250': 0.15789473684210525,
   'precision_at_250': 0.012}},
 'raw_session_baseline': {'average_precision': 0.016299095451800225,
  'reviews_to_first_malicious': 128,
  'random_expected_reviews_to_first': 72.9,
  'ranking_at_k_metrics': {'found_at_25': 0,
   'recall_at_25': 0.0,
   'precision_at_25': 0.0,
   'found_at_50': 0,
   'recall_at_50': 0.0,
   'precision_at_50': 0.0,
   'found_at_100': 0,
   'recall_at_100': 0.0,
   'precision_at_100': 0.0,
   'found_at_250': 2,
   'recall_at_250': 0.10526315789473684,
   'precision_at_250': 

In [81]:
results["slice_manifest"]

,source_file,row_count,start_timestamps,end_timestamps,num_of_processes,num_of_malicious_processes,malicious_users_hosts,benign_users_hosts,session_inactivity_minutes,max_session_duration_minutes,k,scaler,feature_builder,git_commit
0,train-process_uber_summary.parquet,100000,2024-09-18 20:45:15.961034+00:00,2024-09-22 23:57:33.487951+00:00,100000,31,2,13,5.0,30.0,15,StandardScaler,raw_session_stats,N/A


### Build Session Knn Operator out of Raw Feature Matrix

In [82]:
from scipy import sparse

def build_session_knn_operator(X, k = 15):
    """Construct a sparse kNN operator for the given feature matrix X.
    Each session is connected to its k nearest neighbors in the supplied
    method representation X (raw features, M1, M2, or M3).
    """

    X = np.asarray(X, dtype=np.float32)
    n = len(X)
    effective_k = min(k, n - 1)

    # Use cKDTree to find the k nearest neighbors for each session
    _, neighbors = cKDTree(X).query(X, k=effective_k + 1, workers=-1)

    neighbors = np.asarray(neighbors)[:, 1:]  # Exclude self (the first neighbor is the point itself)

    rows = np.repeat(np.arange(n), effective_k)
    cols = neighbors.reshape(-1)
    values = np.ones(len(rows), dtype=float)

    # Binary directed kNN adjacency
    A = sparse.csr_matrix((values, (rows, cols)), shape=(n, n))

    #
    A = A.maximum(A.T)  # Make it symmetric (undirected)
    A.setdiag(0)  # Remove self-loops
    A.eliminate_zeros()  # Remove explicit zeros

    # Row-normalized propagation operator P = D^(-1) A
    degree = np.asarray(A.sum(axis=1)).ravel()
    inverse_degree = np.divide(
        1.0,
        degree,
        out=np.zeros_like(degree),
        where=degree > 0,
    )

    P = sparse.diags(inverse_degree) @ A

    return A, P


### Apply Filter Bank

In [83]:
def session_score_filter_bank(s0, P, restart_alpha=0.5, steps=3):
    s0 = np.asarray(s0, dtype=float)

    p1 = np.asarray(P @ s0).ravel()
    p2 = np.asarray(P @ p1).ravel()

    # Restart diffusion: preserve original evidence while propagating to locally
    diffusion = s0.copy()
    
    for _ in range(steps):
        diffusion = restart_alpha * s0 + (1.0 - restart_alpha) * np.asarray(P @ diffusion).ravel()

    return {
        "raw_knn_distance": s0,
        "neighbour-support": p1,
        "local_residual": s0 - p1,
        "one_vs_two_hop": p1 - p2,
        "restart_diffusion": diffusion,
    }

In [84]:
sessionized_df, sessions = make_strawman_sessions(dev_df)
session_table = build_raw_session_stats(sessionized_df)

feature_columns = [c for c in session_table.columns if c not in {"session_id", "label"}]

In [85]:
assert_canonical_sessions(
    sessionized_df,
    sessions,
    session_table,
)

In [86]:
X = StandardScaler().fit_transform(session_table[feature_columns])

s0 = knn_distance_anomaly_scores(X, k=15)
A, P = build_session_knn_operator(X, k=15)
filter_scores = session_score_filter_bank(s0, P)

y_true = session_table["label"].map({"malicious": 1, "benign": 0}).to_numpy()

filter_results = {
    name: evaluate_precomputed_scores(y_true, scores)
    for name, scores in filter_scores.items()
}

filter_summary = metric_summary_from_method_results(filter_results)
filter_summary

,method,average_precision,reviews_to_first_malicious,random_expected_reviews_to_first,found_at_25,recall_at_25,precision_at_25,found_at_50,recall_at_50,precision_at_50,found_at_100,recall_at_100,precision_at_100,found_at_250,recall_at_250,precision_at_250
0,raw_knn_distance,0.016299,128,72.9,0,0.0,0.0,0,0.0,0.0,0,0.000000,0.00,2,0.105263,0.008
1,neighbour-support,0.022011,120,72.9,0,0.0,0.0,0,0.0,0.0,0,0.000000,0.00,7,0.368421,0.028
2,local_residual,0.009252,272,72.9,0,0.0,0.0,0,0.0,0.0,0,0.000000,0.00,0,0.000000,0.000
3,one_vs_two_hop,0.019823,56,72.9,0,0.0,0.0,0,0.0,0.0,4,0.210526,0.04,7,0.368421,0.028
4,restart_diffusion,0.019326,131,72.9,0,0.0,0.0,0,0.0,0.0,0,0.000000,0.00,3,0.157895,0.012


In [87]:
for method_name, scores in filter_scores.items():
    assert_method_output(session_table, scores)

assert_method_alignment(
    session_table,
    session_table["session_id"].to_numpy(),
    filter_scores["raw_knn_distance"],
)

In [88]:
degree = np.asarray(A.sum(axis=1)).ravel()
row_sums = np.asarray(P.sum(axis=1)).ravel()

n_components, component_labels = connected_components(
    A,
    directed=False,
)

non_isolates = degree > 0
assert np.allclose(row_sums[non_isolates], 1.0)
assert np.allclose(row_sums[~non_isolates], 0.0)

similarity_graph_summary = pd.DataFrame([{
    "nodes": A.shape[0],
    "edges": A.nnz // 2,
    "components": n_components,
    "isolates": int((degree == 0).sum()),
    "degree_min": degree.min(),
    "degree_median": np.median(degree),
    "degree_90th_percentile": np.quantile(degree, 0.90),
    "degree_max": degree.max(),
}])

similarity_graph_summary

,nodes,edges,components,isolates,degree_min,degree_median,degree_90th_percentile,degree_max
0,1457,15308,1,0,15.0,20.0,28.0,53.0


In [89]:
import subprocess

git_commit = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

slice_manifest.loc[0, "num_of_sessions"] = len(session_table)
slice_manifest.loc[0, "num_of_malicious_sessions"] = (
    session_table["label"] == "malicious"
).sum()
slice_manifest.loc[0, "num_session_features"] = len(feature_columns)
slice_manifest.loc[0, "git_commit"] = git_commit

slice_manifest.loc[0, "similarity_graph_nodes"] = A.shape[0]
slice_manifest.loc[0, "similarity_graph_edges"] = A.nnz // 2
slice_manifest.loc[0, "similarity_graph_components"] = n_components
slice_manifest.loc[0, "similarity_graph_isolates"] = (
    degree == 0
).sum()

slice_manifest.loc[0, "filter_restart_alpha"] = 0.5
slice_manifest.loc[0, "filter_steps"] = 3

slice_manifest.to_csv(
    "../artifacts/descriptions/development_slice_manifest.csv",
    index=False,
)

slice_manifest

,source_file,row_count,start_timestamps,end_timestamps,num_of_processes,num_of_malicious_processes,malicious_users_hosts,benign_users_hosts,session_inactivity_minutes,max_session_duration_minutes,...,git_commit,num_of_sessions,num_of_malicious_sessions,num_session_features,similarity_graph_nodes,similarity_graph_edges,similarity_graph_components,similarity_graph_isolates,filter_restart_alpha,filter_steps
0,train-process_uber_summary.parquet,100000,2024-09-18 20:45:15.961034+00:00,2024-09-22 23:57:33.487951+00:00,100000,31,2,13,5.0,30.0,...,2606420,1457.0,19.0,138.0,1457.0,15308.0,1.0,0.0,0.5,3.0


### Null Metrics

In [90]:
import networkx as nx

NULL_METRICS = (
    "average_precision",
    "reviews_to_first_malicious",
    "recall_at_100",
    "recall_at_250",
)

METRIC_DIRECTION = {
    "average_precision": "higher",
    "reviews_to_first_malicious": "lower",
    "recall_at_100": "higher",
    "recall_at_250": "higher",
}


def compact_metrics(y_true, scores):
    result = evaluate_precomputed_scores(y_true, scores)
    ranking = result["ranking_at_k_metrics"]

    return {
        "average_precision": result["average_precision"],
        "reviews_to_first_malicious": result["reviews_to_first_malicious"],
        "recall_at_100": ranking["recall_at_100"],
        "recall_at_250": ranking["recall_at_250"],
    }


def summarize_null(observed_scores, y_true, null_runs):
    rows = []

    for method in null_runs["method"].unique():
        observed = compact_metrics(y_true, observed_scores[method])
        method_null = null_runs[null_runs["method"] == method]

        for metric in NULL_METRICS:
            null_values = method_null[metric].dropna().to_numpy()
            observed_value = observed[metric]

            if METRIC_DIRECTION[metric] == "higher":
                extreme = np.sum(null_values >= observed_value)
            else:
                extreme = np.sum(null_values <= observed_value)

            rows.append({
                "method": method,
                "metric": metric,
                "observed": observed_value,
                "null_mean": null_values.mean(),
                "null_median": np.median(null_values),
                "null_95th_percentile": np.quantile(null_values, 0.95),
                "empirical_p": (extreme + 1) / (len(null_values) + 1),
            })

    return pd.DataFrame(rows)

In [91]:
def label_permutation_null(
    filter_scores,
    y_true,
    n_permutations=1000,
    seed=42,
):
    rng = np.random.default_rng(seed)
    rows = []

    for permutation in range(n_permutations):
        permuted_labels = rng.permutation(y_true)

        for method, scores in filter_scores.items():
            rows.append({
                "replicate": permutation,
                "method": method,
                **compact_metrics(permuted_labels, scores),
            })

    return pd.DataFrame(rows)


label_null_runs = label_permutation_null(
    filter_scores,
    y_true,
    n_permutations=1000,
)

label_null_summary = summarize_null(
    filter_scores,
    y_true,
    label_null_runs,
)

label_null_summary

,method,metric,observed,null_mean,null_median,null_95th_percentile,empirical_p
0,raw_knn_distance,average_precision,0.016299,0.018013,0.015548,0.034119,0.416583
1,raw_knn_distance,reviews_to_first_malicious,128.000000,71.649000,51.500000,213.000000,0.837163
2,raw_knn_distance,recall_at_100,0.000000,0.071316,0.052632,0.157895,1.000000
3,raw_knn_distance,recall_at_250,0.105263,0.174579,0.157895,0.315789,0.869131
4,neighbour-support,average_precision,0.022011,0.018303,0.015201,0.037065,0.166833
5,neighbour-support,reviews_to_first_malicious,120.000000,70.294000,50.000000,196.250000,0.822178
6,neighbour-support,recall_at_100,0.000000,0.070737,0.052632,0.157895,1.000000
7,neighbour-support,recall_at_250,0.368421,0.172895,0.157895,0.315789,0.033966
8,local_residual,average_precision,0.009252,0.018072,0.015346,0.034044,0.999001
9,local_residual,reviews_to_first_malicious,272.000000,70.119000,50.000000,218.150000,0.983017


In [92]:
def row_normalize_adjacency(A):
    degree = np.asarray(A.sum(axis=1)).ravel()

    inverse_degree = np.divide(
        1.0,
        degree,
        out=np.zeros_like(degree, dtype=float),
        where=degree > 0,
    )

    return sparse.diags(inverse_degree) @ A


def degree_preserving_rewire(
    A,
    seed,
    swaps_per_edge=5,
):
    binary_A = (A > 0).astype(np.int8)

    G = nx.from_scipy_sparse_array(
        binary_A,
        create_using=nx.Graph,
    )

    rewired = G.copy()
    n_edges = rewired.number_of_edges()
    n_swaps = swaps_per_edge * n_edges

    nx.double_edge_swap(
        rewired,
        nswap=n_swaps,
        max_tries=20 * n_swaps,
        seed=seed,
    )

    A_rewired = nx.to_scipy_sparse_array(
        rewired,
        nodelist=np.arange(A.shape[0]),
        format="csr",
        dtype=float,
    )

    # Confirm that every node retained its degree.
    original_degree = np.asarray(A.sum(axis=1)).ravel()
    rewired_degree = np.asarray(A_rewired.sum(axis=1)).ravel()
    assert np.array_equal(original_degree, rewired_degree)

    return A_rewired


def graph_rewiring_null(
    A,
    s0,
    y_true,
    n_rewirings=200,
    seed=42,
):
    rng = np.random.default_rng(seed)
    rows = []

    for replicate in range(n_rewirings):
        rewire_seed = int(rng.integers(0, 2**31 - 1))

        A_null = degree_preserving_rewire(
            A,
            seed=rewire_seed,
            swaps_per_edge=5,
        )
        P_null = row_normalize_adjacency(A_null)

        null_scores = session_score_filter_bank(
            s0,
            P_null,
            restart_alpha=0.5,
            steps=3,
        )

        for method, scores in null_scores.items():
            # The raw score does not depend on graph edges.
            if method == "raw_knn_distance":
                continue

            rows.append({
                "replicate": replicate,
                "method": method,
                **compact_metrics(y_true, scores),
            })

    return pd.DataFrame(rows)


graph_null_runs = graph_rewiring_null(
    A,
    s0,
    y_true,
    n_rewirings=200,
)

graph_null_summary = summarize_null(
    filter_scores,
    y_true,
    graph_null_runs,
)

graph_null_summary

KeyboardInterrupt: 

In [ ]:
def filter_bank_selection_test(observed_scores, y_true, null_runs):
    available_methods = list(null_runs["method"].unique())
    observed = {
        method: compact_metrics(y_true, observed_scores[method])
        for method in available_methods
    }

    rows = []

    for metric in NULL_METRICS:
        direction = METRIC_DIRECTION[metric]

        observed_values = {
            method: values[metric]
            for method, values in observed.items()
        }

        if direction == "higher":
            best_method = max(observed_values, key=observed_values.get)
            null_best = null_runs.groupby("replicate")[metric].max()
            extreme = np.sum(null_best >= observed_values[best_method])
        else:
            best_method = min(observed_values, key=observed_values.get)
            null_best = null_runs.groupby("replicate")[metric].min()
            extreme = np.sum(null_best <= observed_values[best_method])

        rows.append({
            "metric": metric,
            "best_observed_method": best_method,
            "best_observed_value": observed_values[best_method],
            "null_best_mean": null_best.mean(),
            "selection_adjusted_p": (
                (extreme + 1) / (len(null_best) + 1)
            ),
        })

    return pd.DataFrame(rows)


graph_bank_test = filter_bank_selection_test(
    filter_scores,
    y_true,
    graph_null_runs,
)

graph_bank_test

,metric,best_observed_method,best_observed_value,null_best_mean,selection_adjusted_p
0,average_precision,neighbour-support,0.022011,0.020633,0.189055
1,reviews_to_first_malicious,one_vs_two_hop,56.000000,56.210000,0.547264
2,recall_at_100,one_vs_two_hop,0.210526,0.074737,0.039801
3,recall_at_250,neighbour-support,0.368421,0.197895,0.039801


In [ ]:
# Stage 5: unified development-set method comparison.
# Deterministic methods retain one row; stochastic methods are summarized
# across the fixed seeds 42, 43, and 44. All declared M0 channels remain
# visible so the table does not select a channel after seeing labels.
COMPARISON_METRICS = [
    "average_precision",
    "recall_at_100",
    "recall_at_250",
    "reviews_to_first_malicious",
]

# Reconfirm that C1/M0 and all telemetry-graph methods use the same
# ordered canonical review units before combining any metric rows.
raw_baseline_session_table = results["session_table"][
    "raw_session_baseline"
]
assert np.array_equal(
    raw_baseline_session_table["session_id"].to_numpy(),
    session_table["session_id"].to_numpy(),
)
assert np.array_equal(
    raw_baseline_session_table["label"].to_numpy(),
    session_table["label"].to_numpy(),
)
assert set(m2_seed_summary["seed"]) == set(EXPERIMENT_SEEDS)
assert set(m3_seed_summary["seed"]) == set(EXPERIMENT_SEEDS)

comparison_rows = []


def add_deterministic_comparison_rows(
    summary,
    family,
    method_names,
    role,
):
    indexed = summary.set_index("method")

    for source_method, display_name in method_names.items():
        values = indexed.loc[source_method]
        row = {
            "family": family,
            "method": display_name,
            "role": role,
            "seed_count": 1,
        }

        for metric in COMPARISON_METRICS:
            row[f"{metric}_mean"] = float(values[metric])
            row[f"{metric}_std"] = np.nan

        comparison_rows.append(row)


def add_stochastic_comparison_rows(
    seed_summary,
    family,
    method_names,
    roles,
):
    for source_method, display_name in method_names.items():
        runs = seed_summary.loc[
            seed_summary["method"] == source_method
        ]
        assert set(runs["seed"]) == set(EXPERIMENT_SEEDS)

        row = {
            "family": family,
            "method": display_name,
            "role": roles[source_method],
            "seed_count": int(runs["seed"].nunique()),
        }

        for metric in COMPARISON_METRICS:
            row[f"{metric}_mean"] = float(runs[metric].mean())
            row[f"{metric}_std"] = float(runs[metric].std())

        comparison_rows.append(row)


add_deterministic_comparison_rows(
    results["metric_summary"],
    family="C0",
    method_names={
        "random_score_baseline": "random ranking",
    },
    role="sanity floor",
)
add_deterministic_comparison_rows(
    results["metric_summary"],
    family="C1",
    method_names={
        "raw_session_baseline": "raw session statistics + kNN",
    },
    role="raw-feature control",
)
add_deterministic_comparison_rows(
    filter_summary,
    family="M0",
    method_names={
        "neighbour-support": "neighbor support",
        "local_residual": "local residual",
        "one_vs_two_hop": "one-vs-two-hop",
        "restart_diffusion": "restart diffusion",
    },
    role="declared filter channel",
)
add_deterministic_comparison_rows(
    m1_summary,
    family="M1",
    method_names={
        "typed_structural_stats_knn": "typed structural statistics + kNN",
    },
    role="primary method",
)
add_stochastic_comparison_rows(
    m2_seed_summary,
    family="M2",
    method_names={
        "node2vec_session_knn": "Node2Vec session representation + kNN",
    },
    roles={
        "node2vec_session_knn": "primary method",
    },
)
add_stochastic_comparison_rows(
    m3_seed_summary,
    family="M3",
    method_names={
        "relational_graphsage_random_init": "GraphSAGE random initialization",
        "relational_graphsage_trained": "GraphSAGE trained",
    },
    roles={
        "relational_graphsage_random_init": "required control",
        "relational_graphsage_trained": "primary method",
    },
)

METHOD_ORDER = {
    "random ranking": 0,
    "raw session statistics + kNN": 1,
    "neighbor support": 2,
    "local residual": 3,
    "one-vs-two-hop": 4,
    "restart diffusion": 5,
    "typed structural statistics + kNN": 6,
    "Node2Vec session representation + kNN": 7,
    "GraphSAGE random initialization": 8,
    "GraphSAGE trained": 9,
}

method_comparison = pd.DataFrame(comparison_rows)
method_comparison["_order"] = method_comparison["method"].map(
    METHOD_ORDER
)
method_comparison = (
    method_comparison
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

assert method_comparison["method"].is_unique
assert method_comparison["family"].tolist() == [
    "C0", "C1", "M0", "M0", "M0", "M0",
    "M1", "M2", "M3", "M3",
]

method_comparison.to_csv(
    "../artifacts/descriptions/method_comparison.csv",
    index=False,
)

method_comparison

,family,method,role,seed_count,average_precision_mean,average_precision_std,recall_at_100_mean,recall_at_100_std,recall_at_250_mean,recall_at_250_std,reviews_to_first_malicious_mean,reviews_to_first_malicious_std
0,C0,random ranking,sanity floor,1,0.014803,NaN,0.052632,NaN,0.157895,NaN,47.000000,NaN
1,C1,raw session statistics + kNN,raw-feature control,1,0.016299,NaN,0.000000,NaN,0.105263,NaN,128.000000,NaN
2,M0,neighbor support,declared filter channel,1,0.022011,NaN,0.000000,NaN,0.368421,NaN,120.000000,NaN
3,M0,local residual,declared filter channel,1,0.009252,NaN,0.000000,NaN,0.000000,NaN,272.000000,NaN
4,M0,one-vs-two-hop,declared filter channel,1,0.019823,NaN,0.210526,NaN,0.368421,NaN,56.000000,NaN
5,M0,restart diffusion,declared filter channel,1,0.019326,NaN,0.000000,NaN,0.157895,NaN,131.000000,NaN
6,M1,typed structural statistics + kNN,primary method,1,0.025201,NaN,0.105263,NaN,0.210526,NaN,7.000000,NaN
7,M2,Node2Vec session representation + kNN,primary method,3,0.066715,0.027104,0.140351,0.030387,0.807018,0.030387,18.333333,25.006666
8,M3,GraphSAGE random initialization,required control,3,0.020189,0.001718,0.087719,0.030387,0.140351,0.030387,73.000000,12.767145
9,M3,GraphSAGE trained,primary method,3,0.019554,0.001625,0.087719,0.030387,0.140351,0.060774,70.333333,24.826062


### Stage 5 — M2 compact source analysis

This bounded analysis checks whether Node2Vec retrieval is explained by session size, executable hubs, degree, or component membership, then compares the real topology with one typed degree-preserving rewire.

In [ ]:
from scipy.stats import spearmanr


# Build label-free session covariates from the exact graph used by M2.
m2_process_session_ids = (
    graph_data["process"].session_id.cpu().numpy()
)
m2_n_processes = graph_data["process"].num_nodes
m2_n_nodes = (
    graph_data["process"].num_nodes
    + graph_data["executable"].num_nodes
)

m2_parent_edges = graph_data[
    "process", "parent_of", "process"
].edge_index.cpu()
m2_executes_edges = graph_data[
    "process", "executes", "executable"
].edge_index.cpu()

m2_parent_in_degree = torch.bincount(
    m2_parent_edges[1],
    minlength=m2_n_processes,
).numpy()
m2_parent_out_degree = torch.bincount(
    m2_parent_edges[0],
    minlength=m2_n_processes,
).numpy()
m2_executable_degree = torch.bincount(
    m2_executes_edges[1],
    minlength=graph_data["executable"].num_nodes,
).numpy()
m2_process_executable_link_count = torch.bincount(
    m2_executes_edges[0],
    minlength=m2_n_processes,
).numpy()
assert m2_process_executable_link_count.max() <= 1

m2_process_executable_degree = np.zeros(m2_n_processes, dtype=float)
m2_process_executable_degree[m2_executes_edges[0].numpy()] = (
    m2_executable_degree[m2_executes_edges[1].numpy()]
)

# Define the two hubs structurally, without consulting labels.
m2_hub_indices = np.argsort(-m2_executable_degree)[:2]
m2_executable_names = {
    executable_index: executable_name
    for executable_name, executable_index
    in builder.executable_index.items()
}
m2_hub_process_flags = {}
m2_hub_audit_rows = []

for hub_rank, hub_index in enumerate(m2_hub_indices, start=1):
    process_nodes = m2_executes_edges[0][
        m2_executes_edges[1] == int(hub_index)
    ].numpy()
    flag = np.zeros(m2_n_processes, dtype=float)
    flag[process_nodes] = 1.0
    m2_hub_process_flags[hub_rank] = flag

    m2_hub_audit_rows.append({
        "hub_rank": hub_rank,
        "executable_index": int(hub_index),
        "executable_name": m2_executable_names[int(hub_index)],
        "degree": int(m2_executable_degree[hub_index]),
        "process_fraction": float(len(process_nodes) / m2_n_processes),
        "sessions_touched": int(np.unique(
            m2_process_session_ids[process_nodes]
        ).size),
    })

m2_hub_audit = pd.DataFrame(m2_hub_audit_rows)

# Components are computed on the exact undirected Node2Vec projection.
m2_projection_rows = node2vec_edge_index[0].cpu().numpy()
m2_projection_cols = node2vec_edge_index[1].cpu().numpy()
m2_projection_adjacency = coo_matrix(
    (
        np.ones(len(m2_projection_rows), dtype=np.int8),
        (m2_projection_rows, m2_projection_cols),
    ),
    shape=(m2_n_nodes, m2_n_nodes),
).tocsr()
m2_component_count, m2_component_labels = connected_components(
    m2_projection_adjacency,
    directed=False,
    return_labels=True,
)
m2_component_sizes = np.bincount(
    m2_component_labels,
    minlength=m2_component_count,
)
m2_largest_component = int(np.argmax(m2_component_sizes))
m2_process_component = m2_component_labels[:m2_n_processes]

m2_process_covariates = pd.DataFrame({
    "session_id": m2_process_session_ids,
    "projection_degree": node2vec_degree[:m2_n_processes].numpy(),
    "executable_degree": m2_process_executable_degree,
    "has_executable_affiliation": (
        m2_process_executable_link_count > 0
    ).astype(float),
    "has_lineage": (
        (m2_parent_in_degree + m2_parent_out_degree) > 0
    ).astype(float),
    "largest_hub": m2_hub_process_flags[1],
    "second_hub": m2_hub_process_flags[2],
    "component": m2_process_component,
    "in_largest_component": (
        m2_process_component == m2_largest_component
    ).astype(float),
})

m2_session_covariates = (
    m2_process_covariates
    .groupby("session_id", sort=True)
    .agg(
        session_size=("session_id", "size"),
        mean_projection_degree=("projection_degree", "mean"),
        max_projection_degree=("projection_degree", "max"),
        mean_executable_degree=("executable_degree", "mean"),
        max_executable_degree=("executable_degree", "max"),
        executable_affiliation_fraction=(
            "has_executable_affiliation", "mean"
        ),
        lineage_process_fraction=("has_lineage", "mean"),
        largest_hub_fraction=("largest_hub", "mean"),
        second_hub_fraction=("second_hub", "mean"),
        component_count=("component", "nunique"),
        largest_component_fraction=(
            "in_largest_component", "mean"
        ),
    )
    .reindex(session_table["session_id"].to_numpy())
    .reset_index()
)
m2_session_covariates["log_session_size"] = np.log1p(
    m2_session_covariates["session_size"]
)
assert len(m2_session_covariates) == len(session_table)
assert m2_session_covariates.isna().sum().sum() == 0
assert np.allclose(
    m2_session_covariates["log_session_size"],
    m1_session_features["log_session_size"],
)

M2_SHORTCUT_COVARIATES = [
    "log_session_size",
    "mean_projection_degree",
    "max_projection_degree",
    "mean_executable_degree",
    "max_executable_degree",
    "executable_affiliation_fraction",
    "lineage_process_fraction",
    "largest_hub_fraction",
    "second_hub_fraction",
    "component_count",
    "largest_component_fraction",
]

m2_shortcut_rows = []
for seed, scores in m2_scores_by_seed.items():
    for covariate in M2_SHORTCUT_COVARIATES:
        values = m2_session_covariates[covariate].to_numpy(dtype=float)
        rho = np.nan
        if np.unique(values).size > 1:
            rho = float(spearmanr(values, scores).statistic)

        m2_shortcut_rows.append({
            "seed": seed,
            "covariate": covariate,
            "spearman_rho": rho,
        })

m2_shortcut_correlations = pd.DataFrame(m2_shortcut_rows)
m2_shortcut_correlation_summary = (
    m2_shortcut_correlations
    .groupby("covariate")["spearman_rho"]
    .agg(
        mean_rho="mean",
        std_rho="std",
        mean_abs_rho=lambda values: np.mean(np.abs(values)),
    )
    .sort_values("mean_abs_rho", ascending=False)
    .reset_index()
)

m2_component_audit = pd.DataFrame([{
    "components": int(m2_component_count),
    "largest_component_nodes": int(m2_component_sizes.max()),
    "largest_component_fraction": float(
        m2_component_sizes.max() / m2_n_nodes
    ),
    "isolates": int((node2vec_degree == 0).sum()),
}])

m2_shortcut_correlations.to_csv(
    "../artifacts/descriptions/m2_shortcut_correlations.csv",
    index=False,
)
display(m2_hub_audit)
display(m2_component_audit)
m2_shortcut_correlation_summary

,hub_rank,executable_index,executable_name,degree,process_fraction,sessions_touched
0,1,0,c:\windows\system32\conhost.exe,74565,0.74565,991
1,2,1,c:\windows\system32\wbem\wmic.exe,17579,0.17579,741


,components,largest_component_nodes,largest_component_fraction,isolates
0,25,93249,0.931996,0


,covariate,mean_rho,std_rho,mean_abs_rho
0,log_session_size,-0.917928,0.002443,0.917928
1,max_executable_degree,-0.757835,0.001068,0.757835
2,second_hub_fraction,-0.721745,0.000922,0.721745
3,component_count,-0.670966,0.005548,0.670966
4,mean_executable_degree,-0.655956,0.003429,0.655956
5,largest_hub_fraction,-0.619977,0.004311,0.619977
6,largest_component_fraction,-0.619703,0.001535,0.619703
7,mean_projection_degree,0.067581,0.004695,0.067581
8,lineage_process_fraction,0.065229,0.005361,0.065229
9,max_projection_degree,-0.053013,0.006729,0.053013


In [ ]:
def rewire_parent_relation(
    edge_index,
    num_nodes,
    seed,
    swaps_per_edge=5,
):
    """Directed double-edge swaps preserving in/out and projection degree."""
    source = edge_index[0].cpu().numpy().copy()
    target = edge_index[1].cpu().numpy().copy()
    n_edges = len(source)
    rng = np.random.default_rng(seed)

    edge_set = set(zip(source.tolist(), target.tolist()))
    undirected_set = {
        tuple(sorted((int(a), int(b))))
        for a, b in edge_set
    }
    assert len(edge_set) == n_edges
    assert len(undirected_set) == n_edges
    assert not np.any(source == target)

    requested_swaps = swaps_per_edge * n_edges
    accepted_swaps = 0
    attempts = 0
    max_attempts = 50 * requested_swaps

    while accepted_swaps < requested_swaps and attempts < max_attempts:
        attempts += 1
        first, second = rng.choice(n_edges, size=2, replace=False)
        a, b = int(source[first]), int(target[first])
        c, d = int(source[second]), int(target[second])

        old_edges = {(a, b), (c, d)}
        new_edges = {(a, d), (c, b)}
        if len(old_edges) < 2 or len(new_edges) < 2:
            continue
        if old_edges == new_edges or a == d or c == b:
            continue
        if any(edge in edge_set - old_edges for edge in new_edges):
            continue

        old_undirected = {
            tuple(sorted(edge)) for edge in old_edges
        }
        new_undirected = {
            tuple(sorted(edge)) for edge in new_edges
        }
        if len(new_undirected) < 2:
            continue
        if any(
            edge in undirected_set - old_undirected
            for edge in new_undirected
        ):
            continue

        edge_set.difference_update(old_edges)
        edge_set.update(new_edges)
        undirected_set.difference_update(old_undirected)
        undirected_set.update(new_undirected)
        target[first], target[second] = d, b
        accepted_swaps += 1

    if accepted_swaps < requested_swaps:
        raise RuntimeError(
            f"Only accepted {accepted_swaps} of {requested_swaps} "
            "requested parent-edge swaps."
        )

    rewired = torch.tensor(
        np.vstack([source, target]),
        dtype=torch.long,
    )
    assert torch.equal(
        torch.bincount(edge_index[0], minlength=num_nodes),
        torch.bincount(rewired[0], minlength=num_nodes),
    )
    assert torch.equal(
        torch.bincount(edge_index[1], minlength=num_nodes),
        torch.bincount(rewired[1], minlength=num_nodes),
    )

    audit = {
        "relation": "parent_of",
        "edges": n_edges,
        "accepted_swaps": accepted_swaps,
        "attempts": attempts,
        "changed_edge_fraction": float(np.mean(
            rewired[1].numpy() != edge_index[1].cpu().numpy()
        )),
    }
    return rewired, audit


def rewire_executes_relation(edge_index, n_processes, n_executables, seed):
    """Permute executable endpoints while preserving both node degrees."""
    source = edge_index[0].cpu()
    target = edge_index[1].cpu()
    source_degree = torch.bincount(source, minlength=n_processes)
    assert int(source_degree.max()) <= 1

    generator = torch.Generator().manual_seed(seed)
    permutation = torch.randperm(len(target), generator=generator)
    rewired = torch.stack([source, target[permutation]])

    assert torch.equal(
        source_degree,
        torch.bincount(rewired[0], minlength=n_processes),
    )
    assert torch.equal(
        torch.bincount(target, minlength=n_executables),
        torch.bincount(rewired[1], minlength=n_executables),
    )
    assert torch.unique(rewired, dim=1).shape[1] == rewired.shape[1]

    audit = {
        "relation": "executes",
        "edges": int(edge_index.shape[1]),
        "accepted_swaps": np.nan,
        "attempts": np.nan,
        "changed_edge_fraction": float(
            (rewired[1] != target).float().mean()
        ),
    }
    return rewired, audit


M2_REWIRE_SEED = 42
m2_rewired_parent_edges, m2_parent_rewire_audit = (
    rewire_parent_relation(
        m2_parent_edges,
        num_nodes=m2_n_processes,
        seed=M2_REWIRE_SEED,
    )
)
m2_rewired_executes_edges, m2_executes_rewire_audit = (
    rewire_executes_relation(
        m2_executes_edges,
        n_processes=m2_n_processes,
        n_executables=graph_data["executable"].num_nodes,
        seed=M2_REWIRE_SEED,
    )
)

m2_rewired_graph = graph_data.clone()
m2_rewired_graph[
    "process", "parent_of", "process"
].edge_index = m2_rewired_parent_edges
m2_rewired_graph[
    "process", "child_of", "process"
].edge_index = m2_rewired_parent_edges.flip(0)
m2_rewired_graph[
    "process", "executes", "executable"
].edge_index = m2_rewired_executes_edges
m2_rewired_graph[
    "executable", "executed_by", "process"
].edge_index = m2_rewired_executes_edges.flip(0)

(
    m2_rewired_node2vec_edge_index,
    m2_rewired_node2vec_degree,
    m2_rewired_projection_summary,
) = build_node2vec_projection(m2_rewired_graph)

# This is a degree-sequence control, including preservation of the hubs.
assert torch.equal(node2vec_degree, m2_rewired_node2vec_degree)
assert m2_rewired_node2vec_edge_index.shape == node2vec_edge_index.shape
m2_rewire_audit = pd.DataFrame([
    m2_parent_rewire_audit,
    m2_executes_rewire_audit,
])
m2_rewire_audit

,relation,edges,accepted_swaps,attempts,changed_edge_fraction
0,parent_of,2318,11590.0,13007.0,0.999569
1,executes,99850,NaN,NaN,0.409835


In [ ]:
m2_rewire_seed_summaries = []
m2_rewire_scores_by_seed = {}
m2_rewire_losses_by_seed = {}
m2_rewired_isolates = m2_rewired_node2vec_degree == 0

for seed in EXPERIMENT_SEEDS:
    rewired_node_embeddings, rewired_losses = train_node2vec(
        m2_rewired_node2vec_edge_index,
        num_nodes=m2_n_nodes,
        seed=seed,
    )
    rewired_node_embeddings[m2_rewired_isolates] = 0.0
    rewired_process_embeddings = rewired_node_embeddings[:m2_n_processes]
    rewired_session_representation = mean_pool_process_embeddings(
        rewired_process_embeddings,
        m2_process_session_ids,
        n_sessions=len(session_table),
    )
    assert_graph_session_representation(
        rewired_session_representation,
        base_dimension=NODE2VEC_CONFIG["embedding_dim"],
    )

    X_m2_rewired = StandardScaler().fit_transform(
        rewired_session_representation.numpy()
    )
    rewired_scores = knn_distance_anomaly_scores(X_m2_rewired, k=K)
    assert_method_alignment(
        session_table,
        session_table["session_id"].to_numpy(),
        rewired_scores,
    )

    rewired_result = evaluate_precomputed_scores(y_true, rewired_scores)
    rewired_summary = metric_summary_from_method_results({
        "node2vec_rewired_session_knn": rewired_result,
    })
    rewired_summary.insert(0, "seed", seed)
    m2_rewire_seed_summaries.append(rewired_summary)
    m2_rewire_scores_by_seed[seed] = rewired_scores
    m2_rewire_losses_by_seed[seed] = rewired_losses

m2_rewire_seed_summary = pd.concat(
    m2_rewire_seed_summaries,
    ignore_index=True,
)

m2_real_for_control = m2_seed_summary.copy()
m2_real_for_control["topology"] = "real"
m2_rewired_for_control = m2_rewire_seed_summary.copy()
m2_rewired_for_control["topology"] = "degree-preserving rewire"
m2_topology_seed_summary = pd.concat(
    [m2_real_for_control, m2_rewired_for_control],
    ignore_index=True,
)

M2_CONTROL_METRICS = [
    "average_precision",
    "recall_at_100",
    "recall_at_250",
    "reviews_to_first_malicious",
]
m2_topology_comparison = (
    m2_topology_seed_summary
    .groupby("topology", sort=False)[M2_CONTROL_METRICS]
    .agg(["mean", "std"])
)

m2_seed_deltas = (
    m2_real_for_control[["seed"] + M2_CONTROL_METRICS]
    .merge(
        m2_rewired_for_control[["seed"] + M2_CONTROL_METRICS],
        on="seed",
        suffixes=("_real", "_rewired"),
        validate="one_to_one",
    )
)
for metric in ("average_precision", "recall_at_100", "recall_at_250"):
    m2_seed_deltas[f"{metric}_delta_real_minus_rewired"] = (
        m2_seed_deltas[f"{metric}_real"]
        - m2_seed_deltas[f"{metric}_rewired"]
    )
m2_seed_deltas["reviews_improvement_real_over_rewired"] = (
    m2_seed_deltas["reviews_to_first_malicious_rewired"]
    - m2_seed_deltas["reviews_to_first_malicious_real"]
)

m2_topology_seed_summary.to_csv(
    "../artifacts/descriptions/m2_rewire_comparison.csv",
    index=False,
)
m2_seed_deltas.to_csv(
    "../artifacts/descriptions/m2_rewire_seed_deltas.csv",
    index=False,
)
display(m2_rewire_seed_summary)
display(m2_seed_deltas)
m2_topology_comparison

seed=42 epoch=01 loss=4.4560
seed=42 epoch=02 loss=3.4080
seed=42 epoch=03 loss=2.8326
seed=42 epoch=04 loss=2.3935
seed=42 epoch=05 loss=2.0528
seed=42 epoch=06 loss=1.7925
seed=42 epoch=07 loss=1.5917
seed=42 epoch=08 loss=1.4357
seed=43 epoch=01 loss=4.4642
seed=43 epoch=02 loss=3.4096
seed=43 epoch=03 loss=2.8301
seed=43 epoch=04 loss=2.3900
seed=43 epoch=05 loss=2.0500
seed=43 epoch=06 loss=1.7923
seed=43 epoch=07 loss=1.5937
seed=43 epoch=08 loss=1.4382
seed=44 epoch=01 loss=4.3499
seed=44 epoch=02 loss=3.3992
seed=44 epoch=03 loss=2.8488
seed=44 epoch=04 loss=2.4034
seed=44 epoch=05 loss=2.0473
seed=44 epoch=06 loss=1.7822
seed=44 epoch=07 loss=1.5843
seed=44 epoch=08 loss=1.4293


,seed,method,average_precision,reviews_to_first_malicious,random_expected_reviews_to_first,found_at_25,recall_at_25,precision_at_25,found_at_50,recall_at_50,precision_at_50,found_at_100,recall_at_100,precision_at_100,found_at_250,recall_at_250,precision_at_250
0,42,node2vec_rewired_session_knn,0.039726,17,72.9,1,0.052632,0.04,1,0.052632,0.02,2,0.105263,0.02,9,0.473684,0.036
1,43,node2vec_rewired_session_knn,0.093011,1,72.9,1,0.052632,0.04,2,0.105263,0.04,2,0.105263,0.02,11,0.578947,0.044
2,44,node2vec_rewired_session_knn,0.046450,9,72.9,1,0.052632,0.04,2,0.105263,0.04,4,0.210526,0.04,9,0.473684,0.036


,seed,average_precision_real,recall_at_100_real,recall_at_250_real,reviews_to_first_malicious_real,average_precision_rewired,recall_at_100_rewired,recall_at_250_rewired,reviews_to_first_malicious_rewired,average_precision_delta_real_minus_rewired,recall_at_100_delta_real_minus_rewired,recall_at_250_delta_real_minus_rewired,reviews_improvement_real_over_rewired
0,42,0.045818,0.105263,0.789474,47,0.039726,0.105263,0.473684,17,0.006093,0.000000,0.315789,-30
1,43,0.097340,0.157895,0.789474,1,0.093011,0.105263,0.578947,1,0.004329,0.052632,0.210526,0
2,44,0.056985,0.157895,0.842105,7,0.046450,0.210526,0.473684,9,0.010536,-0.052632,0.368421,2


average_precision           recall_at_100            \
                                      mean       std          mean       std   
topology                                                                       
real                              0.066715  0.027104      0.140351  0.030387   
degree-preserving rewire          0.059729  0.029019      0.140351  0.060774   

                         recall_at_250           reviews_to_first_malicious  \
                                  mean       std                       mean   
topology                                                                      
real                          0.807018  0.030387                  18.333333   
degree-preserving rewire      0.508772  0.060774                   9.000000   

                                     
                                std  
topology                             
real                      25.006666  
degree-preserving rewire   8.000000

In [ ]:
import json
from IPython.display import Markdown


# Stage 5 localization summary and frozen Stage 6 decision.
m2_real_mean = m2_seed_summary[M2_CONTROL_METRICS].mean()
m2_real_std = m2_seed_summary[M2_CONTROL_METRICS].std()
m2_rewire_mean = m2_rewire_seed_summary[M2_CONTROL_METRICS].mean()
m2_rewire_std = m2_rewire_seed_summary[M2_CONTROL_METRICS].std()

m2_mean_neighbor_purity = float(
    m2_diagnostic_summary["malicious_neighbor_purity_at_k"].mean()
)
m2_base_neighbor_rate = float(
    m2_diagnostic_summary[
        "base_malicious_rate_excluding_self"
    ].mean()
)
m2_size_rho = float(
    m2_shortcut_correlation_summary.loc[
        m2_shortcut_correlation_summary["covariate"]
        == "log_session_size",
        "mean_rho",
    ].iloc[0]
)
m0_one_vs_two_hop_recall_100 = float(
    method_comparison.loc[
        method_comparison["method"] == "one-vs-two-hop",
        "recall_at_100_mean",
    ].iloc[0]
)

m3_random_ap = float(
    m3_seed_summary.loc[
        m3_seed_summary["method"]
        == "relational_graphsage_random_init",
        "average_precision",
    ].mean()
)
m3_trained_ap = float(
    m3_seed_summary.loc[
        m3_seed_summary["method"]
        == "relational_graphsage_trained",
        "average_precision",
    ].mean()
)

assert m2_mean_neighbor_purity < m2_base_neighbor_rate
assert m2_real_mean["recall_at_250"] > m2_rewire_mean["recall_at_250"]
assert np.isclose(
    m2_real_mean["recall_at_100"],
    m2_rewire_mean["recall_at_100"],
)

localization_summary = pd.DataFrame([{
    "stage": 5,
    "status": "complete",
    "selected_family": "M2",
    "selected_representation": "Node2Vec session representation + kNN",
    "selection_scope": "development-informed; frozen before holdout",
    "selection_basis": (
        "highest mean average precision and recall@250 among declared "
        "representations; M0 one-vs-two-hop remains best at recall@100"
    ),
    "signal_regime": (
        "dispersed, size/degree/hub-sensitive short-walk anomalies"
    ),
    "semantic_support": (
        "partial: real topology consistently improves AP modestly and "
        "recall@250 substantially, but not mean recall@100, over the "
        "degree-preserving rewire"
    ),
    "node2vec_ap_mean": float(m2_real_mean["average_precision"]),
    "node2vec_ap_std": float(m2_real_std["average_precision"]),
    "node2vec_recall_at_100_mean": float(
        m2_real_mean["recall_at_100"]
    ),
    "node2vec_recall_at_100_std": float(
        m2_real_std["recall_at_100"]
    ),
    "node2vec_recall_at_250_mean": float(
        m2_real_mean["recall_at_250"]
    ),
    "node2vec_recall_at_250_std": float(
        m2_real_std["recall_at_250"]
    ),
    "m0_best_recall_at_100": m0_one_vs_two_hop_recall_100,
    "rewire_ap_mean": float(m2_rewire_mean["average_precision"]),
    "rewire_recall_at_100_mean": float(
        m2_rewire_mean["recall_at_100"]
    ),
    "rewire_recall_at_250_mean": float(
        m2_rewire_mean["recall_at_250"]
    ),
    "log_session_size_spearman_rho": m2_size_rho,
    "malicious_neighbor_purity_at_15": m2_mean_neighbor_purity,
    "random_neighbor_rate": m2_base_neighbor_rate,
    "graphsage_training_value": (
        "none demonstrated: trained AP does not exceed random-init AP"
    ),
    "graphsage_random_init_ap_mean": m3_random_ap,
    "graphsage_trained_ap_mean": m3_trained_ap,
}])

amplification_comparison = pd.DataFrame([{
    "stage": 6,
    "status": "complete — valid no-amplification outcome",
    "selected_representation": "M2 Node2Vec",
    "amplifier": "none",
    "decision": "do_not_amplify",
    "representation_before": "Node2Vec + log-session-size + kNN",
    "representation_after": "unchanged",
    "decision_reason": (
        "malicious-neighbor purity is below its random base rate; "
        "evidence is dispersed rather than shared/smooth, and much of AP "
        "and recall@100 survives degree-preserving rewiring"
    ),
    "amplification_run": False,
    "real_ap_mean": float(m2_real_mean["average_precision"]),
    "real_recall_at_100_mean": float(
        m2_real_mean["recall_at_100"]
    ),
    "real_recall_at_250_mean": float(
        m2_real_mean["recall_at_250"]
    ),
    "rewire_ap_mean": float(m2_rewire_mean["average_precision"]),
    "rewire_recall_at_100_mean": float(
        m2_rewire_mean["recall_at_100"]
    ),
    "rewire_recall_at_250_mean": float(
        m2_rewire_mean["recall_at_250"]
    ),
    "decision_frozen_before_holdout": True,
}])

frozen_final_config = {
    "freeze_point": "after Stage 6 and before holdout evaluation",
    "review_unit": {
        "identity": "user with host fallback",
        "inactivity_minutes": 5,
        "maximum_duration_minutes": 30,
    },
    "graph": {
        "process_nodes": "one per evaluation process event",
        "parent_relation": "resolved parent_pid_hash to pid_hash",
        "executable_identity": "filename",
        "executable_min_degree": 2,
        "executable_max_degree": None,
        "reverse_traversal_edges": True,
    },
    "representation": {
        "method": "Node2Vec",
        **NODE2VEC_CONFIG,
        "seeds": list(EXPERIMENT_SEEDS),
        "pooling": "process mean plus explicit log1p(session_size)",
        "isolate_handling": "zero process embedding before pooling",
    },
    "scoring": {
        "scaler": "StandardScaler fit label-free per dataset",
        "method": "mean k-nearest-neighbor distance",
        "k": K,
    },
    "amplification": {
        "decision": "do_not_amplify",
        "amplifier": None,
    },
    "holdout": {
        "status": "not evaluated",
        "policy": (
            "refit label-free components once under this frozen config, "
            "then reveal labels and do not revise"
        ),
    },
}

localization_summary.to_csv(
    "../artifacts/descriptions/localization_summary.csv",
    index=False,
)
amplification_comparison.to_csv(
    "../artifacts/descriptions/amplification_comparison.csv",
    index=False,
)
with open(
    "../artifacts/descriptions/frozen_final_config.json",
    "w",
    encoding="utf-8",
) as config_file:
    json.dump(frozen_final_config, config_file, indent=2)

stage5_stage6_summary = f"""
## Stage 5 localization — complete

**Selected representation:** M2 Node2Vec session representation with the
fixed log-session-size channel and kNN scorer.

- Node2Vec has the strongest mean AP ({m2_real_mean['average_precision']:.4f})
  and recall@250 ({m2_real_mean['recall_at_250']:.4f}).
- M0 one-vs-two-hop remains strongest at recall@100
  ({m0_one_vs_two_hop_recall_100:.4f}); Node2Vec reaches
  {m2_real_mean['recall_at_100']:.4f}.
- The real graph improves recall@250 over the degree-preserving rewire
  ({m2_real_mean['recall_at_250']:.4f} versus
  {m2_rewire_mean['recall_at_250']:.4f}), but mean recall@100 is unchanged.
- Node2Vec scores are strongly size-sensitive
  (Spearman rho = {m2_size_rho:.3f}). Semantic support is therefore partial,
  not a claim that topology explains the entire gain.
- Trained GraphSAGE adds no demonstrated value over random initialization.

## Stage 6 amplification — complete

**Decision: do not amplify.** Mean malicious-neighbor purity
({m2_mean_neighbor_purity:.4f}) is below the random neighbor rate
({m2_base_neighbor_rate:.4f}), indicating dispersed rather than smooth/shared
evidence. Diffusion or neighbor support is not scientifically justified for
the selected representation. The unchanged M2 pipeline is frozen before
holdout evaluation.
"""

display(Markdown(stage5_stage6_summary))
display(localization_summary)
amplification_comparison


## Stage 5 localization — complete

**Selected representation:** M2 Node2Vec session representation with the
fixed log-session-size channel and kNN scorer.

- Node2Vec has the strongest mean AP (0.0667)
  and recall@250 (0.8070).
- M0 one-vs-two-hop remains strongest at recall@100
  (0.2105); Node2Vec reaches
  0.1404.
- The real graph improves recall@250 over the degree-preserving rewire
  (0.8070 versus
  0.5088), but mean recall@100 is unchanged.
- Node2Vec scores are strongly size-sensitive
  (Spearman rho = -0.918). Semantic support is therefore partial,
  not a claim that topology explains the entire gain.
- Trained GraphSAGE adds no demonstrated value over random initialization.

## Stage 6 amplification — complete

**Decision: do not amplify.** Mean malicious-neighbor purity
(0.0058) is below the random neighbor rate
(0.0124), indicating dispersed rather than smooth/shared
evidence. Diffusion or neighbor support is not scientifically justified for
the selected representation. The unchanged M2 pipeline is frozen before
holdout evaluation.


,stage,status,selected_family,selected_representation,selection_scope,selection_basis,signal_regime,semantic_support,node2vec_ap_mean,node2vec_ap_std,...,m0_best_recall_at_100,rewire_ap_mean,rewire_recall_at_100_mean,rewire_recall_at_250_mean,log_session_size_spearman_rho,malicious_neighbor_purity_at_15,random_neighbor_rate,graphsage_training_value,graphsage_random_init_ap_mean,graphsage_trained_ap_mean
0,5,complete,M2,Node2Vec session representation + kNN,development-informed; frozen before holdout,highest mean average precision and recall@250 ...,"dispersed, size/degree/hub-sensitive short-wal...",partial: real topology consistently improves A...,0.066715,0.027104,...,0.210526,0.059729,0.140351,0.508772,-0.917928,0.005848,0.012363,none demonstrated: trained AP does not exceed ...,0.020189,0.019554


,stage,status,selected_representation,amplifier,decision,representation_before,representation_after,decision_reason,amplification_run,real_ap_mean,real_recall_at_100_mean,real_recall_at_250_mean,rewire_ap_mean,rewire_recall_at_100_mean,rewire_recall_at_250_mean,decision_frozen_before_holdout
0,6,complete — valid no-amplification outcome,M2 Node2Vec,none,do_not_amplify,Node2Vec + log-session-size + kNN,unchanged,malicious-neighbor purity is below its random ...,False,0.066715,0.140351,0.807018,0.059729,0.140351,0.508772,True


In [ ]:
# Stage 7A: label-blind holdout scoring under the frozen configuration.
# This cell deliberately removes the label column before sessionization,
# graph construction, representation learning, scaling, or scoring.
import hashlib


HOLDOUT_SOURCE_PATH = "../Data/test-process_uber_summary.parquet"
HOLDOUT_SCORE_PATH = (
    "../artifacts/descriptions/holdout_scores_frozen.npz"
)
HOLDOUT_BLIND_MANIFEST_PATH = (
    "../artifacts/descriptions/holdout_blind_manifest.json"
)

assert frozen_final_config["amplification"]["decision"] == "do_not_amplify"
assert frozen_final_config["holdout"]["status"] == "not evaluated"

holdout_blind_df = (
    load_acme_slice(HOLDOUT_SOURCE_PATH, rows=ROWS)
    .drop(columns=["red_team"], errors="ignore")
)
assert "red_team" not in holdout_blind_df.columns

holdout_sessionized_blind, holdout_sessions = make_strawman_sessions(
    holdout_blind_df
)
holdout_raw_session_table = (
    build_raw_session_stats(holdout_sessionized_blind)
    .drop(columns=["label"], errors="ignore")
)
holdout_session_ids = holdout_raw_session_table["session_id"].to_numpy()
assert holdout_raw_session_table["session_id"].is_unique
assert np.array_equal(
    holdout_session_ids,
    np.arange(len(holdout_sessions)),
)
assert len(holdout_sessionized_blind) == len(holdout_blind_df)

# Freeze the raw-feature schema as well as the algorithm.
development_raw_feature_columns = [
    column
    for column in session_table.columns
    if column not in {"session_id", "label"}
]
holdout_raw_feature_columns = [
    column
    for column in holdout_raw_session_table.columns
    if column != "session_id"
]
assert holdout_raw_feature_columns == development_raw_feature_columns

# C0 and C1 controls are scored without labels.
holdout_c0_scores = random_anomaly_scores(
    len(holdout_raw_session_table),
    seed=SEED,
)
holdout_X_raw = StandardScaler().fit_transform(
    holdout_raw_session_table[holdout_raw_feature_columns]
)
holdout_c1_scores = knn_distance_anomaly_scores(holdout_X_raw, k=K)
assert_method_output(holdout_raw_session_table, holdout_c0_scores)
assert_method_output(holdout_raw_session_table, holdout_c1_scores)

# Build the frozen telemetry graph independently on holdout.
holdout_process_feature_columns = strawman_numeric_columns(
    holdout_sessionized_blind
)
development_process_feature_columns = strawman_numeric_columns(
    sessionized_df
)
assert holdout_process_feature_columns == development_process_feature_columns

holdout_builder = TelemetryGraphBuilder()
holdout_graph_data = holdout_builder.build(
    holdout_sessionized_blind,
    feature_cols=holdout_process_feature_columns,
)
assert holdout_graph_data["process"].num_nodes == len(
    holdout_sessionized_blind
)
assert np.array_equal(
    holdout_graph_data["process"].session_id.cpu().numpy(),
    holdout_sessionized_blind["session_id"].to_numpy(),
)

(
    holdout_node2vec_edge_index,
    holdout_node2vec_degree,
    holdout_node2vec_projection_summary,
) = build_node2vec_projection(holdout_graph_data)
holdout_n_processes = holdout_graph_data["process"].num_nodes
holdout_n_nodes = (
    holdout_graph_data["process"].num_nodes
    + holdout_graph_data["executable"].num_nodes
)
holdout_process_session_ids = (
    holdout_graph_data["process"].session_id.cpu()
)
holdout_expected_session_sizes = torch.bincount(
    holdout_process_session_ids,
    minlength=len(holdout_sessions),
).to(torch.float)

holdout_m2_scores_by_seed = {}
holdout_m2_losses_by_seed = {}
holdout_isolates = holdout_node2vec_degree == 0

for seed in EXPERIMENT_SEEDS:
    holdout_node_embeddings, holdout_losses = train_node2vec(
        holdout_node2vec_edge_index,
        num_nodes=holdout_n_nodes,
        seed=seed,
    )
    holdout_node_embeddings[holdout_isolates] = 0.0
    holdout_process_embeddings = holdout_node_embeddings[
        :holdout_n_processes
    ]
    holdout_session_representation = mean_pool_process_embeddings(
        holdout_process_embeddings,
        holdout_process_session_ids,
        n_sessions=len(holdout_sessions),
    )
    assert holdout_session_representation.shape == (
        len(holdout_sessions),
        NODE2VEC_CONFIG["embedding_dim"] + 1,
    )
    assert torch.allclose(
        holdout_session_representation[:, -1],
        torch.log1p(holdout_expected_session_sizes),
        atol=1e-6,
        rtol=1e-6,
    )

    holdout_X_m2 = StandardScaler().fit_transform(
        holdout_session_representation.numpy()
    )
    holdout_scores = knn_distance_anomaly_scores(holdout_X_m2, k=K)
    assert_method_alignment(
        holdout_raw_session_table,
        holdout_session_ids,
        holdout_scores,
    )
    holdout_m2_scores_by_seed[seed] = holdout_scores
    holdout_m2_losses_by_seed[seed] = holdout_losses

# Persist only IDs and label-free scores before the evaluation cell exists.
holdout_score_arrays = {
    "session_id": holdout_session_ids.astype(np.int64),
    "c0_random_seed_42": np.asarray(holdout_c0_scores, dtype=np.float64),
    "c1_raw_session_knn": np.asarray(holdout_c1_scores, dtype=np.float64),
}
for seed, scores in holdout_m2_scores_by_seed.items():
    holdout_score_arrays[f"m2_node2vec_seed_{seed}"] = np.asarray(
        scores,
        dtype=np.float64,
    )

holdout_score_hash = hashlib.sha256()
for array_name in sorted(holdout_score_arrays):
    holdout_score_hash.update(array_name.encode("utf-8"))
    holdout_score_hash.update(
        np.ascontiguousarray(holdout_score_arrays[array_name]).tobytes()
    )
holdout_score_sha256 = holdout_score_hash.hexdigest()
np.savez_compressed(HOLDOUT_SCORE_PATH, **holdout_score_arrays)

frozen_config_sha256 = hashlib.sha256(
    json.dumps(
        frozen_final_config,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

holdout_parent_edges = holdout_graph_data[
    "process", "parent_of", "process"
].edge_index.shape[1]
holdout_executes_edges = holdout_graph_data[
    "process", "executes", "executable"
].edge_index.shape[1]
holdout_blind_manifest = {
    "status": "scores frozen before label construction",
    "source_file": os.path.basename(HOLDOUT_SOURCE_PATH),
    "rows": int(len(holdout_blind_df)),
    "start_timestamp": str(holdout_blind_df["_time"].min()),
    "end_timestamp": str(holdout_blind_df["_time"].max()),
    "sessions": int(len(holdout_sessions)),
    "raw_session_features": int(len(holdout_raw_feature_columns)),
    "process_features": int(len(holdout_process_feature_columns)),
    "process_nodes": int(holdout_n_processes),
    "executable_nodes": int(
        holdout_graph_data["executable"].num_nodes
    ),
    "parent_edges": int(holdout_parent_edges),
    "executes_edges": int(holdout_executes_edges),
    "projection_isolates": int(holdout_isolates.sum()),
    "score_file": os.path.basename(HOLDOUT_SCORE_PATH),
    "score_sha256": holdout_score_sha256,
    "frozen_config_sha256": frozen_config_sha256,
    "methods_scored": [
        "C0 random seed 42",
        "C1 raw session statistics + kNN",
        "M2 Node2Vec seeds 42, 43, 44",
    ],
    "label_columns_available_during_scoring": [],
}
with open(
    HOLDOUT_BLIND_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(holdout_blind_manifest, manifest_file, indent=2)

display(pd.DataFrame([holdout_blind_manifest]))
holdout_node2vec_projection_summary

seed=42 epoch=01 loss=4.3199
seed=42 epoch=02 loss=3.3965
seed=42 epoch=03 loss=2.8680
seed=42 epoch=04 loss=2.4146
seed=42 epoch=05 loss=2.0520
seed=42 epoch=06 loss=1.7807
seed=42 epoch=07 loss=1.5772
seed=42 epoch=08 loss=1.4221
seed=43 epoch=01 loss=4.4241
seed=43 epoch=02 loss=3.4007
seed=43 epoch=03 loss=2.8409
seed=43 epoch=04 loss=2.4017
seed=43 epoch=05 loss=2.0528
seed=43 epoch=06 loss=1.7857
seed=43 epoch=07 loss=1.5857
seed=43 epoch=08 loss=1.4291
seed=44 epoch=01 loss=4.3873
seed=44 epoch=02 loss=3.3949
seed=44 epoch=03 loss=2.8378
seed=44 epoch=04 loss=2.3873
seed=44 epoch=05 loss=2.0428
seed=44 epoch=06 loss=1.7830
seed=44 epoch=07 loss=1.5823
seed=44 epoch=08 loss=1.4277


,status,source_file,rows,start_timestamp,end_timestamp,sessions,raw_session_features,process_features,process_nodes,executable_nodes,parent_edges,executes_edges,projection_isolates,score_file,score_sha256,frozen_config_sha256,methods_scored,label_columns_available_during_scoring
0,scores frozen before label construction,test-process_uber_summary.parquet,100000,2024-09-07 17:52:33.408538+00:00,2024-09-09 06:59:58.960918+00:00,1074,138,67,100000,45,1087,99868,3,holdout_scores_frozen.npz,7e12491752872e44f8e3d576c834fd0a12eb41d1413b90...,e205351455ca405eccdf0338d2a2f94039846594ca4b7b...,"[C0 random seed 42, C1 raw session statistics ...",[]


,nodes,process_nodes,executable_nodes,directed_walk_edges,undirected_relations,isolates
0,100045,100000,45,201910,100955,3


In [ ]:
# Stage 7B: one-time holdout label reveal and evaluation.
# Do not run this cell until Stage 7A has completed and its score hash exists.
HOLDOUT_RESULT_PATH = (
    "../artifacts/descriptions/holdout_result.csv"
)
HOLDOUT_RUNS_PATH = (
    "../artifacts/descriptions/holdout_metric_runs.csv"
)
HOLDOUT_EVALUATION_MANIFEST_PATH = (
    "../artifacts/descriptions/holdout_evaluation_manifest.json"
)

if os.path.exists(HOLDOUT_RESULT_PATH):
    raise RuntimeError(
        "holdout_result.csv already exists. The one-time holdout gate is "
        "closed; do not reevaluate or revise the frozen method."
    )

with open(
    HOLDOUT_BLIND_MANIFEST_PATH,
    encoding="utf-8",
) as manifest_file:
    frozen_holdout_manifest = json.load(manifest_file)

with np.load(HOLDOUT_SCORE_PATH) as frozen_score_file:
    frozen_holdout_scores = {
        name: frozen_score_file[name].copy()
        for name in frozen_score_file.files
    }

verification_hash = hashlib.sha256()
for array_name in sorted(frozen_holdout_scores):
    verification_hash.update(array_name.encode("utf-8"))
    verification_hash.update(
        np.ascontiguousarray(frozen_holdout_scores[array_name]).tobytes()
    )
assert verification_hash.hexdigest() == frozen_holdout_manifest[
    "score_sha256"
]

# Labels enter for the first time in the frozen holdout workflow here.
holdout_labeled_df = load_acme_slice(HOLDOUT_SOURCE_PATH, rows=ROWS)
holdout_sessionized_labeled, holdout_labeled_sessions = (
    make_strawman_sessions(holdout_labeled_df)
)
holdout_labeled_session_table = build_raw_session_stats(
    holdout_sessionized_labeled
)

assert np.array_equal(
    holdout_sessionized_labeled["pid_hash"].to_numpy(),
    holdout_sessionized_blind["pid_hash"].to_numpy(),
)
assert np.array_equal(
    holdout_sessionized_labeled["session_id"].to_numpy(),
    holdout_sessionized_blind["session_id"].to_numpy(),
)
assert np.array_equal(
    frozen_holdout_scores["session_id"],
    holdout_labeled_session_table["session_id"].to_numpy(),
)

holdout_y_true = (
    holdout_labeled_session_table["label"]
    .map({"benign": 0, "malicious": 1})
    .to_numpy(dtype=int)
)
assert holdout_y_true.sum() > 0

holdout_run_rows = []


def add_holdout_metric_run(method, role, seed, scores):
    assert_method_output(holdout_labeled_session_table, scores)
    metrics = evaluate_precomputed_scores(holdout_y_true, scores)
    summary = metric_summary_from_method_results({method: metrics}).iloc[0]
    holdout_run_rows.append({
        "method": method,
        "role": role,
        "seed": seed,
        **{
            metric: float(summary[metric])
            for metric in COMPARISON_METRICS
        },
    })


add_holdout_metric_run(
    "C0 random ranking",
    "sanity floor",
    SEED,
    frozen_holdout_scores["c0_random_seed_42"],
)
add_holdout_metric_run(
    "C1 raw session statistics + kNN",
    "raw-feature control",
    np.nan,
    frozen_holdout_scores["c1_raw_session_knn"],
)
for seed in EXPERIMENT_SEEDS:
    add_holdout_metric_run(
        "M2 Node2Vec session representation + kNN",
        "frozen selected method",
        seed,
        frozen_holdout_scores[f"m2_node2vec_seed_{seed}"],
    )

holdout_metric_runs = pd.DataFrame(holdout_run_rows)
holdout_result_rows = []
for (method, role), runs in holdout_metric_runs.groupby(
    ["method", "role"],
    sort=False,
):
    row = {
        "method": method,
        "role": role,
        "run_count": int(len(runs)),
    }
    for metric in COMPARISON_METRICS:
        row[f"{metric}_mean"] = float(runs[metric].mean())
        row[f"{metric}_std"] = (
            float(runs[metric].std()) if len(runs) > 1 else np.nan
        )
    holdout_result_rows.append(row)

holdout_result = pd.DataFrame(holdout_result_rows)
holdout_metric_runs.to_csv(HOLDOUT_RUNS_PATH, index=False)
holdout_result.to_csv(HOLDOUT_RESULT_PATH, index=False)

holdout_evaluation_manifest = {
    "status": "evaluated once; method frozen",
    "sessions": int(len(holdout_labeled_session_table)),
    "malicious_sessions": int(holdout_y_true.sum()),
    "score_sha256": frozen_holdout_manifest["score_sha256"],
    "frozen_config_sha256": frozen_holdout_manifest[
        "frozen_config_sha256"
    ],
    "selection_or_tuning_after_evaluation": False,
    "blinding_note": (
        "Scoring and score persistence were label-blind. The source test "
        "dataframe had been loaded earlier in the exploratory notebook, "
        "so this is procedurally separated rather than a cryptographically "
        "unseen external holdout."
    ),
}
with open(
    HOLDOUT_EVALUATION_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as evaluation_file:
    json.dump(holdout_evaluation_manifest, evaluation_file, indent=2)

display(Markdown(
    "## Stage 7 — frozen holdout evaluated once\n\n"
    f"Sessions: **{len(holdout_labeled_session_table):,}**; "
    f"malicious sessions: **{int(holdout_y_true.sum()):,}**.\n\n"
    "No method selection, amplification, or parameter revision is "
    "permitted after this result."
))
holdout_result

RuntimeError: holdout_result.csv already exists. The one-time holdout gate is closed; do not reevaluate or revise the frozen method.

## Signal Diagnosis

In [93]:
def compute_signal_diagnostic_channels(P, s):
    """Compute graph-relative channels from intrinsic anomaly scores.

    Parameters
    ----------
    P : scipy.sparse matrix
        Row-normalized session propagation operator built from X.
    s : np.ndarray
        Intrinsic kNN-distance anomaly scores, one per session.

    Returns
    -------
    dict[str, np.ndarray]
        Intrinsic, neighbor-supported, residual, and pocket score channels.
    """
    s = np.asarray(s, dtype=float).ravel()
    p1s = np.asarray(P @ s).ravel()
    p2s = np.asarray(P @ p1s).ravel()

    return {
        "intrinsic": s,
        "neighbor_support": p1s,
        "local_contrast": s - p1s,
        "local_contrast_magnitude": np.abs(s - p1s),
        "pocket": p1s - p2s,
        "supported_candidate": 0.5 * s + 0.5 * p1s,
    }


def apply_matched_amplification(s, channels, regime, alpha=0.5):
    """Return at most one score transform matched to a diagnosed regime."""
    s = np.asarray(s, dtype=float).ravel()
    if regime == "Neighborhood-supported":
        return alpha * s + (1.0 - alpha) * channels["neighbor_support"]
    if regime == "Locally Contrastive":
        return channels["local_contrast"]
    if regime == "Intermediate-scale":
        return channels["pocket"]
    if regime == "No useful organization":
        return s
    raise ValueError(f"Unknown signal regime: {regime}")

### Assemble every method representation for signal diagnosis

This is the canonical handoff from method evaluation to signal analysis. It constructs the scaled session matrix $X$ and intrinsic kNN-distance anomaly signal $s$ for every representation in one place. The random-rank control is intentionally absent because it has no feature representation $X$.

In [94]:
# Canonical method artifacts: scaled session representation X and intrinsic score s.
# Model fitting and session pooling happen in the method sections above; all signal-
# analysis inputs are standardized and scored together here to prevent mismatches.

SIGNAL_SCORE_K = K
SIGNAL_GRAPH_K = K
signal_method_artifacts = {}


def register_signal_method(method_key, method_name, representation, dimension_names):
    """Standardize one session representation and create its signal artifacts."""
    if hasattr(representation, "detach"):
        representation = representation.detach().cpu().numpy()

    representation = np.asarray(representation, dtype=np.float32)
    if representation.ndim != 2:
        raise ValueError(f"{method_key}: representation must be a 2D matrix")
    if representation.shape[0] != len(session_table):
        raise ValueError(
            f"{method_key}: {representation.shape[0]} rows for "
            f"{len(session_table)} sessions"
        )

    scaler = StandardScaler()
    X = scaler.fit_transform(representation)
    s = knn_distance_anomaly_scores(X, k=SIGNAL_SCORE_K)

    # Build the session-similarity graph from the same X used to compute s.
    A, P = build_session_knn_operator(X, k=SIGNAL_GRAPH_K)
    channels = compute_signal_diagnostic_channels(P, s)

    assert X.shape[0] == len(s) == A.shape[0] == A.shape[1]
    signal_method_artifacts[method_key] = {
        "method": method_name,
        "session_ids": session_table["session_id"].to_numpy(copy=True),
        "dimension_names": list(dimension_names),
        "scaler": scaler,
        "X": X,
        "s": s,
        "A": A,
        "P": P,
        "channels": channels,
    }


# C1: raw session-statistic representation.
register_signal_method(
    method_key="c1_raw_session_knn",
    method_name="C1 raw session statistics + kNN",
    representation=session_table[feature_columns],
    dimension_names=feature_columns,
)

# M1: pooled structural-feature representation.
register_signal_method(
    method_key="m1_structural_knn",
    method_name="M1 structural session representation + kNN",
    representation=m1_session_features[m1_feature_columns],
    dimension_names=m1_feature_columns,
)

# M2: one pooled Node2Vec session representation per experimental seed.
for seed in EXPERIMENT_SEEDS:
    m2_representation = m2_embeddings_by_seed[seed]
    register_signal_method(
        method_key=f"m2_node2vec_seed_{seed}",
        method_name="M2 Node2Vec session representation + kNN",
        representation=m2_representation,
        dimension_names=[
            f"node2vec_{column_index:03d}"
            for column_index in range(m2_representation.shape[1])
        ],
    )

# M3: random-init and trained GraphSAGE representations for every seed.
for (seed, variant), m3_representation in sorted(m3_embeddings_by_seed.items()):
    register_signal_method(
        method_key=f"m3_graphsage_{variant}_seed_{seed}",
        method_name=f"M3 GraphSAGE {variant} session representation + kNN",
        representation=m3_representation,
        dimension_names=[
            f"graphsage_{column_index:03d}"
            for column_index in range(m3_representation.shape[1])
        ],
    )

signal_method_summary = pd.DataFrame(
    [
        {
            "method_key": method_key,
            "method": artifact["method"],
            "sessions": artifact["X"].shape[0],
            "dimensions": artifact["X"].shape[1],
            "score_min": float(np.min(artifact["s"])),
            "score_median": float(np.median(artifact["s"])),
            "score_max": float(np.max(artifact["s"])),
        }
        for method_key, artifact in signal_method_artifacts.items()
    ]
)

signal_method_summary

,method_key,method,sessions,dimensions,score_min,score_median,score_max
0,c1_raw_session_knn,C1 raw session statistics + kNN,1457,138,0.539361,1.603409,239.544019
1,m1_structural_knn,M1 structural session representation + kNN,1457,6,0.000000,0.026317,30.701960
2,m2_node2vec_seed_42,M2 Node2Vec session representation + kNN,1457,49,1.036183,1.759430,18.039038
3,m2_node2vec_seed_43,M2 Node2Vec session representation + kNN,1457,49,1.063551,1.762527,18.619192
4,m2_node2vec_seed_44,M2 Node2Vec session representation + kNN,1457,49,1.017403,1.733054,19.647344
5,m3_graphsage_random_init_seed_42,M3 GraphSAGE random_init session representatio...,1457,49,0.014454,0.075427,16.996349
6,m3_graphsage_trained_seed_42,M3 GraphSAGE trained session representation + kNN,1457,49,0.010488,0.056312,16.435756
7,m3_graphsage_random_init_seed_43,M3 GraphSAGE random_init session representatio...,1457,49,0.015586,0.088770,23.240315
8,m3_graphsage_trained_seed_43,M3 GraphSAGE trained session representation + kNN,1457,49,0.009435,0.056771,18.089389
9,m3_graphsage_random_init_seed_44,M3 GraphSAGE random_init session representatio...,1457,49,0.012385,0.071568,17.280744


In [95]:
def session_score_filter_bank(s0, P, restart_alpha=0.5, steps=3):
    s0 = np.asarray(s0, dtype=float)

    p1 = np.asarray(P @ s0).ravel()
    p2 = np.asarray(P @ p1).ravel()

    # Restart diffusion: preserve original evidence while propagating to locally
    diffusion = s0.copy()
    
    for _ in range(steps):
        diffusion = restart_alpha * s0 + (1.0 - restart_alpha) * np.asarray(P @ diffusion).ravel()

    return {
        "raw_knn_distance": s0,
        "neighbour-support": p1,
        "local_residual": s0 - p1,
        "one_vs_two_hop": p1 - p2,
        "restart_diffusion": diffusion,
    }

In [96]:
selected = signal_method_artifacts["m2_node2vec_seed_42"]

X = selected["X"]
s = selected["s"]
P = selected["P"]
channels = selected["channels"]

In [100]:
selected

{'method': 'M2 Node2Vec session representation + kNN',
 'session_ids': array([   0,    1,    2, ..., 1454, 1455, 1456], shape=(1457,)),
 'dimension_names': ['node2vec_000',
  'node2vec_001',
  'node2vec_002',
  'node2vec_003',
  'node2vec_004',
  'node2vec_005',
  'node2vec_006',
  'node2vec_007',
  'node2vec_008',
  'node2vec_009',
  'node2vec_010',
  'node2vec_011',
  'node2vec_012',
  'node2vec_013',
  'node2vec_014',
  'node2vec_015',
  'node2vec_016',
  'node2vec_017',
  'node2vec_018',
  'node2vec_019',
  'node2vec_020',
  'node2vec_021',
  'node2vec_022',
  'node2vec_023',
  'node2vec_024',
  'node2vec_025',
  'node2vec_026',
  'node2vec_027',
  'node2vec_028',
  'node2vec_029',
  'node2vec_030',
  'node2vec_031',
  'node2vec_032',
  'node2vec_033',
  'node2vec_034',
  'node2vec_035',
  'node2vec_036',
  'node2vec_037',
  'node2vec_038',
  'node2vec_039',
  'node2vec_040',
  'node2vec_041',
  'node2vec_042',
  'node2vec_043',
  'node2vec_044',
  'node2vec_045',
  'node2vec_046',

In [105]:
# Pick the signal regime for the selected representation without using labels.
# The null breaks the alignment between session scores and the session graph while
# preserving the graph, score distribution, and number of investigated sessions.

REGIME_NAMES = (
    "Neighborhood-supported",
    "Locally Contrastive",
    "Intermediate-scale",
)


def _signal_regime_statistics(P, s, tail_fraction=0.05, min_tail_size=10):
    """Return one targeted statistic for each candidate signal regime."""
    s = np.asarray(s, dtype=float).ravel()
    if len(s) < 2 or not np.all(np.isfinite(s)):
        raise ValueError("s must contain at least two finite anomaly scores")
    if not 0 < tail_fraction <= 1:
        raise ValueError("tail_fraction must be in (0, 1]")

    score_std = float(np.std(s))
    if score_std <= np.finfo(float).eps:
        raise ValueError("Signal diagnosis requires non-constant anomaly scores")

    # Standardization makes statistics comparable across representations without
    # changing session rankings or the graph topology.
    z = (s - np.mean(s)) / score_std
    p1 = np.asarray(P @ z).ravel()
    p2 = np.asarray(P @ p1).ravel()

    tail_size = min(
        len(z),
        max(int(min_tail_size), int(np.ceil(tail_fraction * len(z)))),
    )
    intrinsic_tail = np.argpartition(z, -tail_size)[-tail_size:]
    neighborhood_tail = np.argpartition(p1, -tail_size)[-tail_size:]

    return np.array(
        [
            # Do intrinsically anomalous sessions also have anomalous neighbors?
            np.mean(p1[intrinsic_tail]),
            # Do intrinsically anomalous sessions rise above normal neighbors?
            np.mean((z - p1)[intrinsic_tail]),
            # Do the strongest 1-hop neighborhoods dilute at the second hop?
            np.mean((p1 - p2)[neighborhood_tail]),
        ],
        dtype=float,
    )


def pick_signal_regime(
    artifact,
    n_permutations=999,
    tail_fraction=0.05,
    min_tail_size=10,
    alpha=0.05,
    random_state=42,
):
    """Diagnose graph organization using a family-wise permutation test.

    A regime is selected only when its observed statistic beats the permutation
    null after correcting for trying all three regimes. Otherwise the correct
    result is ``No useful organization``.
    """
    if n_permutations < 99:
        raise ValueError("Use at least 99 permutations for a meaningful null")
    if not 0 < alpha < 1:
        raise ValueError("alpha must be in (0, 1)")

    P = artifact["P"]
    s = np.asarray(artifact["s"], dtype=float).ravel()
    if P.shape != (len(s), len(s)):
        raise ValueError("P and s are not aligned")

    observed = _signal_regime_statistics(
        P,
        s,
        tail_fraction=tail_fraction,
        min_tail_size=min_tail_size,
    )

    rng = np.random.default_rng(random_state)
    null_statistics = np.empty((n_permutations, len(REGIME_NAMES)))
    for permutation_index in range(n_permutations):
        null_statistics[permutation_index] = _signal_regime_statistics(
            P,
            rng.permutation(s),
            tail_fraction=tail_fraction,
            min_tail_size=min_tail_size,
        )

    null_mean = null_statistics.mean(axis=0)
    null_std = null_statistics.std(axis=0, ddof=1)
    safe_null_std = np.where(
        null_std > np.finfo(float).eps,
        null_std,
        np.inf,
    )
    effect_z = (observed - null_mean) / safe_null_std

    # Raw one-sided permutation p-values answer each hypothesis separately.
    raw_p = (
        1 + np.sum(null_statistics >= observed[None, :], axis=0)
    ) / (n_permutations + 1)

    # Max-statistic correction controls false selection across the three regimes.
    standardized_null = (null_statistics - null_mean) / safe_null_std
    max_null = np.max(standardized_null, axis=1)
    adjusted_p = np.array(
        [
            (1 + np.sum(max_null >= candidate_z)) / (n_permutations + 1)
            for candidate_z in effect_z
        ]
    )
    passes = (effect_z > 0) & (adjusted_p <= alpha)

    evidence = pd.DataFrame(
        {
            "regime": REGIME_NAMES,
            "observed_statistic": observed,
            "null_mean": null_mean,
            "null_std": null_std,
            "effect_z": effect_z,
            "raw_p": raw_p,
            "familywise_p": adjusted_p,
            "passes": passes,
        }
    )

    if np.any(passes):
        eligible_indices = np.flatnonzero(passes)
        winner_index = eligible_indices[np.argmax(effect_z[eligible_indices])]
        regime = REGIME_NAMES[winner_index]
    else:
        winner_index = None
        regime = "No useful organization"

    return {
        "regime": regime,
        "winner_index": winner_index,
        "evidence": evidence,
        "n_permutations": n_permutations,
        "tail_fraction": tail_fraction,
        "alpha": alpha,
    }


signal_regime_result = pick_signal_regime(selected)
selected["signal_regime_result"] = signal_regime_result
signal_regime = signal_regime_result["regime"]
signal_regime_evidence = signal_regime_result["evidence"]

display(Markdown(f"### Diagnosed signal regime: **{signal_regime}**"))
signal_regime_evidence.style.format(
    {
        "observed_statistic": "{:.3f}",
        "null_mean": "{:.3f}",
        "null_std": "{:.3f}",
        "effect_z": "{:.2f}",
        "raw_p": "{:.4f}",
        "familywise_p": "{:.4f}",
    }
)

### Diagnosed signal regime: **Intermediate-scale**

,regime,observed_statistic,null_mean,null_std,effect_z,raw_p,familywise_p,passes
0,Neighborhood-supported,0.084,-0.002,0.034,2.51,0.0060,0.0270,True
1,Locally Contrastive,2.167,2.253,0.034,-2.51,0.9950,1.0000,False
2,Intermediate-scale,0.616,0.440,0.029,6.12,0.0010,0.0010,True


In [110]:
selected["signal_regime_result"]

amplified_scores = apply_matched_amplification(
    selected["s"],
    selected["channels"],
    regime=signal_regime,
    alpha=0.5,
)

amplified_scores

array([-0.47748857, -0.4669188 , -0.25115904, ..., -1.87225353,
       -3.15767102, -1.28925495], shape=(1457,))

In [111]:
signal_results = pd.DataFrame({
    "session_id": selected["session_ids"],
    "intrinsic_score": selected["s"],
    "amplified_score": amplified_scores,
})

signal_results["intrinsic_rank"] = (
    signal_results["intrinsic_score"]
    .rank(ascending=False, method="min")
)

signal_results["amplified_rank"] = (
    signal_results["amplified_score"]
    .rank(ascending=False, method="min")
)

# Positive means amplification moved the session closer to the top.
signal_results["rank_gain"] = (
    signal_results["intrinsic_rank"]
    - signal_results["amplified_rank"]
)

signal_results["signal_regime"] = signal_regime

signal_results = signal_results.sort_values(
    "amplified_score",
    ascending=False,
)

signal_results.head(25)

,session_id,intrinsic_score,amplified_score,intrinsic_rank,amplified_rank,rank_gain,signal_regime
241,241,4.160703,5.423874,508.0,1.0,507.0,Intermediate-scale
68,68,3.617727,4.291916,541.0,2.0,539.0,Intermediate-scale
215,215,3.335678,4.217553,561.0,3.0,558.0,Intermediate-scale
49,49,2.938055,4.204683,600.0,4.0,596.0,Intermediate-scale
177,177,3.473444,4.161777,551.0,5.0,546.0,Intermediate-scale
199,199,3.693666,4.143128,533.0,6.0,527.0,Intermediate-scale
56,56,4.501153,4.111225,494.0,7.0,487.0,Intermediate-scale
123,123,4.375146,4.039777,499.0,8.0,491.0,Intermediate-scale
1211,1211,2.272303,4.037222,686.0,9.0,677.0,Intermediate-scale
172,172,2.999499,4.011918,595.0,10.0,585.0,Intermediate-scale


In [112]:
diagnosis_y_true = (
    session_table["label"]
    .map({"benign": 0, "malicious": 1})
    .to_numpy(dtype=int)
)

amplification_results = {
    "original_m2_knn": evaluate_precomputed_scores(
        diagnosis_y_true,
        selected["s"],
    ),
    "matched_intermediate_scale": evaluate_precomputed_scores(
        diagnosis_y_true,
        amplified_scores,
    ),
}

amplification_summary = metric_summary_from_method_results(
    amplification_results
)

amplification_summary

,method,average_precision,reviews_to_first_malicious,random_expected_reviews_to_first,found_at_25,recall_at_25,precision_at_25,found_at_50,recall_at_50,precision_at_50,found_at_100,recall_at_100,precision_at_100,found_at_250,recall_at_250,precision_at_250
0,original_m2_knn,0.045818,47,72.9,0,0.0,0.0,1,0.052632,0.02,2,0.105263,0.02,15,0.789474,0.06
1,matched_intermediate_scale,0.007705,379,72.9,0,0.0,0.0,0,0.000000,0.00,0,0.000000,0.00,0,0.000000,0.00


In [113]:
pocket_details = pd.DataFrame({
    "session_id": selected["session_ids"],
    "s": selected["s"],
    "Ps": selected["channels"]["neighbor_support"],
    "P2s": (
        selected["channels"]["neighbor_support"]
        - selected["channels"]["pocket"]
    ),
    "pocket_score": selected["channels"]["pocket"],
}).sort_values("pocket_score", ascending=False)

pocket_details.head(25)

,session_id,s,Ps,P2s,pocket_score
241,241,4.160703,10.382175,4.958301,5.423874
68,68,3.617727,9.144164,4.852248,4.291916
215,215,3.335678,7.715586,3.498032,4.217553
49,49,2.938055,7.356100,3.151417,4.204683
177,177,3.473444,6.764768,2.602991,4.161777
199,199,3.693666,8.157688,4.014560,4.143128
56,56,4.501153,8.757253,4.646028,4.111225
123,123,4.375146,7.459881,3.420104,4.039777
1211,1211,2.272303,8.295246,4.258024,4.037222
172,172,2.999499,7.438910,3.426993,4.011918


In [103]:
from IPython.display import display, Markdown

In [115]:
# Interactive signal-regime dashboard for the selected representation.
# Left: where each signal channel lives in the M2 representation.
# Right: how the pocket transform reordered sessions relative to intrinsic kNN.

import altair as alt

signal_umap = UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    metric="euclidean",
    random_state=SEED,
).fit_transform(selected["X"])

signal_plot_data = pd.DataFrame(
    {
        "session_id": selected["session_ids"],
        "umap_x": signal_umap[:, 0],
        "umap_y": signal_umap[:, 1],
        "intrinsic_score": selected["s"],
        "neighbor_score": selected["channels"]["neighbor_support"],
        "two_hop_score": (
            selected["channels"]["neighbor_support"]
            - selected["channels"]["pocket"]
        ),
        "pocket_score": selected["channels"]["pocket"],
    }
)

signal_plot_data = signal_plot_data.merge(
    session_table[["session_id", "label"]],
    on="session_id",
    how="left",
    validate="one_to_one",
)

signal_plot_data["intrinsic_rank"] = signal_plot_data[
    "intrinsic_score"
].rank(ascending=False, method="min")
signal_plot_data["pocket_rank"] = signal_plot_data[
    "pocket_score"
].rank(ascending=False, method="min")
signal_plot_data["rank_gain"] = (
    signal_plot_data["intrinsic_rank"]
    - signal_plot_data["pocket_rank"]
)

# Percentiles let one shared color scale work for channels with different units.
score_columns = (
    "intrinsic_score",
    "neighbor_score",
    "two_hop_score",
    "pocket_score",
)
for score_column in score_columns:
    signal_plot_data[f"{score_column}_percentile"] = signal_plot_data[
        score_column
    ].rank(pct=True, method="average")

score_selector = alt.param(
    name="score_field",
    value="pocket_score_percentile",
    bind=alt.binding_select(
        name="Color map by: ",
        options=[f"{column}_percentile" for column in score_columns],
        labels=["Intrinsic s", "Neighbor Ps", "Two-hop P²s", "Pocket Ps − P²s"],
    ),
)
map_brush = alt.selection_point(
    name="inspect_sessions",
    fields=["session_id"],
    on="click",
    toggle="event.shiftKey",
    empty=True,
)

tooltip_fields = [
    alt.Tooltip("session_id:N", title="Session"),
    alt.Tooltip("label:N", title="Development label"),
    alt.Tooltip("intrinsic_score:Q", title="s", format=".3f"),
    alt.Tooltip("neighbor_score:Q", title="Ps", format=".3f"),
    alt.Tooltip("two_hop_score:Q", title="P²s", format=".3f"),
    alt.Tooltip("pocket_score:Q", title="Ps − P²s", format=".3f"),
    alt.Tooltip("intrinsic_rank:Q", title="Intrinsic rank", format=".0f"),
    alt.Tooltip("pocket_rank:Q", title="Pocket rank", format=".0f"),
    alt.Tooltip("rank_gain:Q", title="Rank gain", format="+.0f"),
]

signal_map = (
    alt.Chart(signal_plot_data)
    .transform_calculate(selected_score="datum[score_field]")
    .mark_point(filled=True, size=58, opacity=0.78)
    .encode(
        x=alt.X("umap_x:Q", title="UMAP 1", axis=None),
        y=alt.Y("umap_y:Q", title="UMAP 2", axis=None),
        color=alt.Color(
            "selected_score:Q",
            title="Score percentile",
            scale=alt.Scale(domain=[0, 1], scheme="viridis"),
        ),
        shape=alt.Shape(
            "label:N",
            title="Development label",
            scale=alt.Scale(
                domain=["benign", "malicious"],
                range=["circle", "triangle-up"],
            ),
        ),
        tooltip=tooltip_fields,
    )
    .add_params(score_selector, map_brush)
    .properties(
        width=600,
        height=540,
        title="M2 signal geometry — click to select, Shift-click for more, scroll to zoom",
    )
    .interactive()
)

rank_points = (
    alt.Chart(signal_plot_data)
    .mark_point(filled=True, size=55)
    .encode(
        x=alt.X("intrinsic_rank:Q", title="Original M2 rank"),
        y=alt.Y(
            "pocket_rank:Q",
            title="Pocket rank",
            scale=alt.Scale(reverse=True),
        ),
        color=alt.Color(
            "label:N",
            title="Development label",
            scale=alt.Scale(
                domain=["benign", "malicious"],
                range=["#9aa0a6", "#d62728"],
            ),
        ),
        opacity=alt.condition(map_brush, alt.value(0.9), alt.value(0.08)),
        tooltip=tooltip_fields,
    )
    .properties(
        width=430,
        height=540,
        title="Re-ranking outcome — malicious sessions are red",
    )
    .interactive()
)

rank_diagonal_data = pd.DataFrame(
    {
        "intrinsic_rank": [1, len(signal_plot_data)],
        "pocket_rank": [1, len(signal_plot_data)],
    }
)
rank_diagonal = (
    alt.Chart(rank_diagonal_data)
    .mark_line(color="#555555", strokeDash=[6, 4], opacity=0.55)
    .encode(x="intrinsic_rank:Q", y="pocket_rank:Q")
)

signal_regime_dashboard = (
    signal_map | (rank_points + rank_diagonal)
).resolve_scale(color="independent")

signal_regime_dashboard

/work/home/mwasti/tuttInstitute/Steering-Telemetry-Triage-with-Self-Supervised-Graph-Geometry/GraphTriage/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/work/home/mwasti/tmp/ipykernel_2182049/887827069.py:162: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  signal_map | (rank_points + rank_diagonal)


alt.HConcatChart(...)